<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/22.Reasoning_and_Acting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 22 — Reasoning and Acting

*Companion notebook untuk* LLM Design Patterns — **edisi Colab GPU**

Misalkan kita bertanya ke model bahasa: *"Berapa penduduk kota Ostmund?"* Ostmund tidak ada di dunia nyata — kotanya karangan notebook ini — jadi model yang hanya mengandalkan ingatan pra-latihnya pasti menebak. Manusia akan melakukan hal lain: berpikir sebentar, membuka ensiklopedia, membaca hasilnya, baru menjawab.

ReAct (*Reasoning and Acting*, Yao dkk. 2022) mengubah kebiasaan itu menjadi loop teks:

- **Thought** — model menulis alasannya, misalnya "saya perlu mencari Ostmund";
- **Action** dan **Action Input** — model memilih alat dan isinya, misalnya `Search` dengan input `Ostmund`;
- **Observation** — program kita menjalankan alat itu dan menempelkan hasilnya ke prompt;
- loop berulang sampai model menulis **Final Answer**.

Bab ini membangun loop itu dengan LangChain dalam tiga gaya — agen bawaan `initialize_agent`, agen *document store*, dan pipeline LCEL (*LangChain Expression Language*) yang dijalankan `AgentExecutor` — lalu membahas evaluasi, keamanan, dan keterbatasannya. Di bagian akhir kita mengukur pertanyaan yang sering terlewat: apakah model benar-benar *membaca* Observation, atau hanya meniru formatnya?

> **Catatan lingkungan — cukup dibaca sekali**
>
> - **Tanpa API berbayar.** Listing buku memanggil OpenAI dengan kunci API. Di sini model bahasa berjalan lokal di GPU: **Qwen2.5-1.5B-Instruct** (model instruksi publik, Apache-2.0) menjadi otak agen, dan **GPT-2 124M** (model dasar tanpa *instruction tuning*) menjadi pembanding. Angka di notebook ini mengukur model kecil tersebut, bukan `gpt-4o`.
> - **Alat terkendali.** Semua alat di jalur utama bersifat lokal dan hanya membaca: ensiklopedia kota fiktif dan kalkulator yang menolak apa pun selain angka dan operator. Tidak ada kode buatan model yang dieksekusi. Contoh Wikipedia dari buku tersedia sebagai opsi (`RUN_WIKIPEDIA`).
> - **Listing buku yang tidak jalan lagi** dengan LangChain sekarang ditaruh di kotak kutipan seperti ini, bersama error yang sudah diverifikasi. Sel kode sesudahnya adalah versi yang bekerja.
> - **Status pengujian.** Notebook ini **belum dijalankan di Colab maupun di GPU CUDA**. Seluruh jalur utamanya (tanpa sel opsional Wikipedia) sudah dijalankan dari kernel bersih di Apple M1 Pro (GPU MPS; model agen FP16 seperti di T4, GPT-2 FP32), dengan sel cek-CUDA dan sel presisi diganti khusus untuk Mac. Angka yang dikutip di teks berasal dari run itu; output sel sengaja dikosongkan. Run itu mencatat 948 panggilan model (4 di antaranya ditolak penjaga konteks) dan selesai dalam sekitar 20 menit; waktu di T4 belum diukur. Di GPU lain, perbedaan numerik kecil bisa mengubah token yang dipilih model: hitungan eksperimen bisa bergeser, dan jejak demo satu soal bisa berbeda sama sekali — misalnya model Anda justru memanggil `Calculator`. Yang perlu dibaca adalah polanya.

## Persiapan

Tiga langkah singkat: memasang paket LangChain yang dikunci versinya, mengatur konfigurasi, lalu memastikan GPU benar-benar bisa dipakai sebelum memuat model.

In [1]:
# Paket LangChain dikunci ke versi yang dipakai saat notebook ini diuji (rilis Oktober 2026).
# torch dan transformers memakai versi bawaan Colab; versinya dicetak dan diperiksa di sel impor.
import sys
import subprocess
import importlib.metadata as md

PINS = {"langchain-core": "1.6.10", "langchain-classic": "1.0.8", "langchain-community": "0.4.2", "numexpr": "2.14.2"}


def installed(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


need = [f"{p}=={v}" for p, v in PINS.items() if installed(p) != v]
if need:
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + proc.stderr[-3000:])
print("terpasang:", " | ".join(f"{p} {installed(p)}" for p in PINS))

terpasang: langchain-core 1.6.10 | langchain-classic 1.0.8 | langchain-community 0.4.2 | numexpr 2.14.2


In [2]:
ALLOW_CPU = False        # True = izinkan lanjut di CPU bila CUDA tidak siap (sangat lambat, belum diuji); jumlah soal dikecilkan
RUN_WIKIPEDIA = False    # True = jalankan juga contoh Wikipedia dari buku (butuh internet, hanya membaca)

AGENT_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # model instruksi publik (Apache-2.0), tanpa token Hugging Face
BASE_MODEL_ID = "openai-community/gpt2"          # GPT-2 124M: model dasar tanpa instruction tuning, sebagai pembanding

N_KB = 30    # soal ensiklopedia (10 per jenis pertanyaan)
N_MA = 20    # soal aritmetika MultiArith

Sel berikut tidak hanya bertanya `torch.cuda.is_available()`. Ia mengalokasikan matriks di GPU, mengalikannya, lalu menunggu GPU selesai (`synchronize`). Kalau salah satu langkah gagal, notebook berhenti dengan petunjuk **Runtime > Change runtime type > GPU** — tidak diam-diam pindah ke CPU, karena seluruh notebook memanggil model hampir seribu kali. CPU hanya dipakai bila `ALLOW_CPU = True`, dan jumlah soal otomatis dikecilkan.

Presisi dipilih dari kemampuan GPU: BF16 untuk GPU generasi Ampere ke atas (*compute capability* ≥ 8), FP16 untuk T4 yang tidak punya BF16 native. GPT-2 tetap FP32 karena ukurannya kecil.

In [3]:
import torch


def cuda_ready():
    """CUDA dianggap siap hanya bila alokasi, perkalian matriks, dan sinkronisasi benar-benar berhasil."""
    if not torch.cuda.is_available():
        return False, "torch.cuda.is_available() = False"
    try:
        x = torch.randn(1024, 1024, device="cuda")
        y = x @ x
        torch.cuda.synchronize()
        finite = bool(torch.isfinite(y).all().item())
    except Exception as err:   # gagal apa pun di uji kecil ini berarti CUDA tidak bisa dipakai
        return False, f"{type(err).__name__}: {err}"
    return (True, "ok") if finite else (False, "hasil perkalian uji tidak finite")


ok, why = cuda_ready()
if ok:
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"GPU {props.name} | compute capability {props.major}.{props.minor} | VRAM bebas {free / 2**30:.1f} dari {total / 2**30:.1f} GiB")
elif ALLOW_CPU:
    DEVICE = torch.device("cpu")
    N_KB, N_MA = 6, 4
    print(f"CUDA tidak siap ({why}). ALLOW_CPU=True: lanjut di CPU dengan {N_KB} soal ensiklopedia dan {N_MA} soal MultiArith. "
          "Qwen2.5-1.5B FP32 di CPU sangat lambat, dan jalur ini belum diuji.")
else:
    raise RuntimeError(
        f"CUDA tidak siap ({why}). Di Colab: Runtime > Change runtime type > pilih GPU (mis. T4), "
        "lalu jalankan ulang dari atas. Untuk sengaja memakai CPU, set ALLOW_CPU = True di sel konfigurasi."
    )

GPU Tesla T4 | compute capability 7.5 | VRAM bebas 14.4 dari 14.6 GiB


In [4]:
if DEVICE.type == "cuda":
    major, minor = torch.cuda.get_device_capability()
    AGENT_DTYPE = torch.bfloat16 if major >= 8 else torch.float16   # T4 (7.5) tidak punya BF16 native -> FP16
else:
    AGENT_DTYPE = torch.float32
BASE_DTYPE = torch.float32   # GPT-2 hanya 124M parameter; FP32 menjaga greedy decoding tetap stabil
print("presisi model agen:", AGENT_DTYPE, "| presisi GPT-2:", BASE_DTYPE)

presisi model agen: torch.float16 | presisi GPT-2: torch.float32


In [5]:
import os
import re
import gc
import sys
import subprocess
import json
import math
import time
import random
import hashlib
import urllib.request
import warnings
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import numexpr
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 56), \
    "butuh transformers >= 4.56: jalankan pip install -U transformers, lalu Runtime > Restart session"
random.seed(0)
torch.manual_seed(0)
WORK = Path("ch22_work")
WORK.mkdir(exist_ok=True)
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}")

python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2 | device cuda


## Dua model, satu antarmuka

Agen ReAct butuh model yang mau mengikuti format. **Qwen2.5-1.5B-Instruct** sudah dilatih mengikuti instruksi, jadi ia menggantikan `ChatOpenAI` di semua listing. **GPT-2** adalah model dasar: ia hanya melanjutkan teks, tanpa pernah dilatih menjawab perintah. Membandingkan keduanya memperlihatkan apa yang sebenarnya dibutuhkan ReAct dari sebuah model.

Setelah dimuat, setiap parameter diperiksa: semuanya harus berada di device yang sama dengan `DEVICE`. Satu forward pass uji memeriksa bahwa logit model agen dalam presisi setengah (FP16 di T4, BF16 di GPU yang lebih baru) tidak berisi `inf`/`NaN`; kalau ada, model dimuat ulang dalam FP32 dan hal itu dicetak. Karena uji itu hanya sampel, `LocalChat` (sel berikutnya) juga berhenti dengan pesan jelas bila keluaran model hanya berisi token `!` — gejala khas logit NaN.

In [6]:
def load_model(model_id, dtype):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(DEVICE).eval()
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    # greedy murni: abaikan sampling/repetition penalty bawaan model (padanan temperature=0 di buku)
    model.generation_config = GenerationConfig(do_sample=False, num_beams=1, repetition_penalty=1.0,
                                               eos_token_id=model.generation_config.eos_token_id, pad_token_id=pad)
    return tok, model


def logits_finite(model, tok):
    """Sampel satu forward pass dengan chat template: logit FP16/BF16 harus finite."""
    text = tok.apply_chat_template([{"role": "user", "content": "Question: What is the population of Ostmund?\nThought:"}],
                                   tokenize=False, add_generation_prompt=True)
    probe = tok(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    with torch.inference_mode():
        return bool(torch.isfinite(model(**probe).logits).all().item())


print("Run pertama mengunduh bobot model (sekitar 3 GB untuk Qwen); peringatan HF_TOKEN boleh diabaikan karena kedua model publik.")
base_tok, base_model = load_model(BASE_MODEL_ID, BASE_DTYPE)
agent_tok, agent_model = load_model(AGENT_MODEL_ID, AGENT_DTYPE)
if not logits_finite(agent_model, agent_tok):
    print(f"Logit model agen tidak finite dalam {AGENT_DTYPE}; dimuat ulang dalam float32.")
    del agent_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    AGENT_DTYPE = torch.float32
    agent_tok, agent_model = load_model(AGENT_MODEL_ID, AGENT_DTYPE)

AGENT_LABEL = AGENT_MODEL_ID.split("/")[-1]
for label, model_id, model in [("GPT-2", BASE_MODEL_ID, base_model), (AGENT_LABEL, AGENT_MODEL_ID, agent_model)]:
    devices = {p.device.type for p in model.parameters()}
    dtypes = {str(p.dtype).replace("torch.", "") for p in model.parameters()}
    assert devices == {DEVICE.type}, f"{label}: parameter tersebar di {devices}"
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{label:<22} {n_params / 1e6:>6,.0f} juta parameter | device {devices.pop()} | dtype {dtypes.pop()}")
if DEVICE.type == "cuda":
    print(f"memori GPU terpakai: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

Run pertama mengunduh bobot model (sekitar 3 GB untuk Qwen); peringatan HF_TOKEN boleh diabaikan karena kedua model publik.
GPT-2                     124 juta parameter | device cuda | dtype float32
Qwen2.5-1.5B-Instruct   1,544 juta parameter | device cuda | dtype float16
memori GPU terpakai: 3.35 GiB


LangChain berbicara dengan model lewat antarmuka *chat model*. `LocalChat` membungkus model Hugging Face agar bisa dipakai di mana pun buku memakai `ChatOpenAI`:

- **Format prompt.** Untuk Qwen, pesan dibungkus *chat template* model itu; untuk GPT-2, teksnya disambung apa adanya.
- **Stop sequence.** Daftar `stop` dari LangChain diteruskan ke `generate`, lalu teks dipotong di stop pertama — perilaku yang sama dengan API OpenAI.
- **Greedy decoding** (`do_sample=False`): model selalu memilih token yang paling mungkin, padanan `temperature=0` di buku. Di perangkat yang sama, hasilnya bisa diulang.
- **Log dan penjaga konteks.** Setiap panggilan dicatat di `CALLS` (token prompt, token baru, teks), dan prompt yang tak muat di konteks ditolak dengan `ContextOverflow` yang jelas.

Input ditokenisasi langsung ke device model, `generate` berjalan di GPU, dan hanya token baru yang dibawa ke CPU untuk di-decode.

In [7]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult

CALLS = []   # satu entri per panggilan model: model, token prompt, token baru, teks mentah, teks setelah stop, detik


class ContextOverflow(Exception):
    """Prompt ditambah token baru tidak muat di jendela konteks yang diizinkan."""


def cut(text, stops):
    """Potong teks di stop sequence pertama, seperti yang dilakukan API chat model."""
    hits = [text.find(s) for s in stops if s in text]
    return text[:min(hits)] if hits else text


class LocalChat(BaseChatModel):
    """Model Hugging Face lokal sebagai chat model LangChain; generate selalu di device model (CUDA)."""

    label: str
    hf_model: Any
    hf_tok: Any
    use_chat_template: bool
    max_context: int
    max_new_tokens: int = 64

    @property
    def _llm_type(self) -> str:
        return "local-hf"

    def render(self, messages):
        if not self.use_chat_template:
            return "\n".join(str(m.content) for m in messages)   # GPT-2 tak punya format chat: teks disambung apa adanya
        roles = {"human": "user", "ai": "assistant", "system": "system"}
        turns = [{"role": roles.get(m.type, "user"), "content": str(m.content)} for m in messages]
        return self.hf_tok.apply_chat_template(turns, tokenize=False, add_generation_prompt=True)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        stop = stop or []
        enc = self.hf_tok(self.render(messages), return_tensors="pt", add_special_tokens=False).to(self.hf_model.device)
        n = enc["input_ids"].shape[1]
        if n + self.max_new_tokens > self.max_context:
            CALLS.append(dict(model=self.label, prompt_tokens=n, new_tokens=0, raw=None, text=None, s=0.0, overflow=True))
            raise ContextOverflow(f"{self.label}: prompt {n} token + {self.max_new_tokens} token baru > konteks {self.max_context}")
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.hf_model.generate(**enc, max_new_tokens=self.max_new_tokens,
                                         stop_strings=stop or None, tokenizer=self.hf_tok)
        new_ids = out[0, n:].cpu()   # hanya token baru yang dibawa ke CPU, untuk di-decode
        if self.hf_model.dtype != torch.float32 and len(new_ids) > 1 and bool((new_ids == 0).all()):
            raise RuntimeError(f"{self.label}: keluaran hanya token id 0 ('!'), gejala logit NaN di {self.hf_model.dtype}; "
                               "set AGENT_DTYPE = torch.float32 lalu jalankan ulang sel model")
        raw = self.hf_tok.decode(new_ids, skip_special_tokens=True)
        text = cut(raw, stop)
        CALLS.append(dict(model=self.label, prompt_tokens=n, new_tokens=len(new_ids), raw=raw, text=text,
                          s=time.perf_counter() - t0, overflow=False))
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])


def base_llm(max_new_tokens=64):
    return LocalChat(label="GPT-2", hf_model=base_model, hf_tok=base_tok, use_chat_template=False,
                     max_context=base_model.config.n_positions, max_new_tokens=max_new_tokens)


def agent_llm(max_new_tokens=128):
    # konteks dibatasi 4.096 token (model mendukung lebih panjang) agar memori dan waktu tetap terkendali
    return LocalChat(label=AGENT_LABEL, hf_model=agent_model, hf_tok=agent_tok, use_chat_template=True,
                     max_context=4096, max_new_tokens=max_new_tokens)


print(agent_llm().invoke("Reply with exactly one word: ready").content)

Ready


## Dunia kecil untuk menguji agen

Supaya jawaban yang benar *hanya* bisa datang dari alat, sel berikut membangkitkan 44 kota fiktif, masing-masing dengan lima fakta karangan: negara, tahun berdiri, sungai, populasi, dan wali kota. Tiga kota menjadi contoh di prompt *few-shot* nanti — prompt yang menyertakan beberapa soal lengkap beserta jawabannya — dan 30 kota lain menjadi soal uji (`N_KB`; 6 bila `ALLOW_CPU`). Sisanya hanya mengisi ensiklopedia. Populasi, tahun, dan wali kota setiap kota uji sengaja dibuat berbeda dari milik kota contoh, sehingga jawaban yang benar tidak mungkin hasil menyalin contoh.

In [8]:
# 44 kota fiktif, nama dan faktanya dibangkitkan dengan seed 0. Karena fiktif, tak ada model yang bisa
# mengetahuinya dari pra-latih: jawaban benar hanya bisa datang dari hasil alat (Observation).
rnd = random.Random(0)
SYL = ["Var", "Quil", "Dra", "Bel", "Sar", "Mor", "Kel", "Fen", "Dal", "Ost", "Rin", "Vel", "Gar", "Thal", "Zun", "Hal", "Pell", "Brin", "Cor", "Lum"]
SUF = ["holt", "mere", "by", "ford", "stad", "wick", "dorn", "vale", "mund", "ton"]
COUNTRIES = ["Ostravia", "Kelmoria", "Dranvel", "Sarnovia", "Belquor", "Thalmund"]
FIRST = ["Elsa", "Tomas", "Ingrid", "Marek", "Liesel", "Anton", "Greta", "Pavel", "Hanna", "Oskar", "Vera", "Emil"]
LAST = ["Morrin", "Halvek", "Ovelin", "Praskov", "Durvane", "Kerrow", "Lindqvar", "Mattrel", "Brevold", "Sollen", "Varquist", "Ostrel"]
RIVERS = ["Quell", "Ambrel", "Silvar", "Tornis", "Everin", "Mirel", "Kaskel", "Lennor"]
names = rnd.sample([a + b for a in SYL for b in SUF], 44)
CITIES = [dict(city=c, country=rnd.choice(COUNTRIES), founded=str(rnd.randint(1150, 1890)), river=rnd.choice(RIVERS),
               population=f"{rnd.randint(12, 950) * 1000 + rnd.randint(0, 999):,}", mayor=f"{rnd.choice(FIRST)} {rnd.choice(LAST)}")
          for c in names]
KB = CITIES[:40]       # kota uji
SHOTS = CITIES[40:]    # kota contoh untuk prompt few-shot
for c in KB:           # nilai kota uji yang kebetulan sama dengan nilai kota contoh dibangkitkan ulang
    while c["founded"] in {s["founded"] for s in SHOTS}:
        c["founded"] = str(rnd.randint(1150, 1890))
    while c["population"] in {s["population"] for s in SHOTS}:
        c["population"] = f"{rnd.randint(12, 950) * 1000 + rnd.randint(0, 999):,}"
    while c["mayor"] in {s["mayor"] for s in SHOTS}:
        c["mayor"] = f"{rnd.choice(FIRST)} {rnd.choice(LAST)}"
SHOTS = SHOTS[:3]
BYNAME = {c["city"]: c for c in CITIES}
ATTR = ["country", "founded", "river", "population", "mayor"]
TYPES = ["population", "founded", "mayor"]   # tiga jenis pertanyaan di eksperimen
QTXT = {"population": "What is the population of {}?", "founded": "In which year was {} founded?", "mayor": "Who is the mayor of {}?"}


def sentences(c):
    return {"country": f"{c['city']} is a city in {c['country']}.", "founded": f"{c['city']} was founded in {c['founded']}.",
            "river": f"{c['city']} lies on the river {c['river']}.", "population": f"The population of {c['city']} is {c['population']}.",
            "mayor": f"The mayor of {c['city']} is {c['mayor']}."}


def flat(c, drop=None):
    """Dokumen satu baris: lima kalimat fakta (atribut `drop` dibuang bila diisi)."""
    return " ".join(sentences(c)[a] for a in ATTR if a != drop)


def paragraphs(c):
    """Dokumen berparagraf untuk DocstoreExplorer (Lookup bekerja per paragraf)."""
    s = sentences(c)
    return f"{s['country']} {s['founded']} {s['river']}\n\n{s['population']}\n\n{s['mayor']}"


print(f"{len(CITIES)} kota, {len(BYNAME)} nama unik | contoh dokumen:\n{flat(KB[0])}")

44 kota, 44 nama unik | contoh dokumen:
Ostmund is a city in Ostravia. Ostmund was founded in 1886. Ostmund lies on the river Kaskel. The population of Ostmund is 739,844. The mayor of Ostmund is Vera Varquist.


Agen butuh alat. Dua alat berikut sengaja dibuat sempit:

- **`Search`** mengembalikan dokumen satu kota, atau *Could not find* bila namanya tidak ada. Spasi, baris baru, dan tanda kutip di tepi input dibuang — kita akan melihat kenapa itu perlu.
- **`Calculator`** menghitung ekspresi aritmetika dengan `numexpr`, tetapi hanya setelah memastikan isinya cuma angka dan operator. Pemisah ribuan seperti `739,844` dibersihkan dulu.

Keduanya mencatat setiap eksekusi di `TOOL_LOG`, sehingga kita bisa membedakan aksi yang hanya *ditulis* model dari aksi yang *benar-benar dijalankan*. Alat, parser, dan statistik di notebook ini sengaja berjalan di CPU: semuanya pekerjaan teks ringan. Yang butuh GPU hanya `generate`.

In [9]:
TOOL_LOG = []   # setiap eksekusi alat yang sungguh terjadi: (nama alat, input)


def fmt(x):
    return str(int(x)) if float(x).is_integer() else f"{x:g}"


def search_city(query):
    """Alat baca-saja: cari satu kota di ensiklopedia lokal. Spasi, baris baru, dan tanda kutip di tepi dibuang."""
    TOOL_LOG.append(("Search", query))
    name = query.strip().strip("\"'").strip()
    return flat(BYNAME[name]) if name in BYNAME else f"Could not find [{name}]."


SAFE_EXPR = re.compile(r"[0-9.+\-*/()\s]+")


def calculator(expression):
    """Alat hitung terkendali: hanya angka dan + - * / ( ); tanpa nama, fungsi, atau kode Python."""
    TOOL_LOG.append(("Calculator", expression))
    expr = re.sub(r"(?<=\d),(?=\d{3})", "", expression.strip())   # 739,844 -> 739844
    if not SAFE_EXPR.fullmatch(expr):
        return "Error: only numbers and + - * / ( ) are allowed"
    if len(expr) > 100:
        return "Error: expression too long"
    power = "**" in expr
    if power:   # pangkat dihitung sebagai float: pangkat raksasa langsung OverflowError, bukan hitungan bilangan bulat tanpa akhir
        expr = re.sub(r"(?<![\d.])(\d+)(?![\d.])", r"\1.0", expr)
    try:
        value = float(numexpr.evaluate(expr, local_dict={}, global_dict={}))
    except (SyntaxError, ValueError, KeyError, TypeError, ZeroDivisionError, OverflowError) as err:
        return f"Error: {type(err).__name__}"
    if not math.isfinite(value) or (power and abs(value) >= 1e15):
        return "Error: OverflowError"
    return fmt(value)


c0 = KB[0]
print(search_city(c0["city"]))
print(search_city("Atlantis"))
print(calculator("2026 - 1886"), "|", calculator("739,844 / 1000"), "|", calculator("__import__('os')"))
print(calculator("1483 ** 2"), "|", calculator("9" * 30 + " * 2"), "|", calculator("9**9**9"))

Ostmund is a city in Ostravia. Ostmund was founded in 1886. Ostmund lies on the river Kaskel. The population of Ostmund is 739,844. The mayor of Ostmund is Vera Varquist.
Could not find [Atlantis].
140 | 739.844 | Error: only numbers and + - * / ( ) are allowed
2199289 | Error: OverflowError | Error: OverflowError


Dokumen Ostmund berisi lima kalimat; populasinya 739,844 dan kota itu berdiri tahun 1886. Nama yang tidak ada dijawab *Could not find*, bukan error, sehingga agen bisa membaca kegagalannya sebagai Observation. Dua baris terakhir menguji kalkulator: `2026 - 1886` menjadi `140`, `739,844 / 1000` menjadi `739.844` (keluaran Python; titiknya titik desimal), `1483 ** 2` menjadi `2199289`, dan upaya menyisipkan kode Python ditolak sebelum menyentuh evaluator. Angka raksasa — tiga puluh angka 9, atau pangkat bertingkat `9**9**9` — langsung dikembalikan sebagai `Error: OverflowError`, bukan exception yang menghentikan loop atau hitungan yang tak pernah selesai. Di dalam loop, galat itu sampai ke model sebagai Observation.

## ReAct dengan tangan, sebelum framework

Sebelum memakai LangChain, kita jalankan satu loop ReAct dengan 20-an baris Python. Tujuannya memisahkan lima hal yang di framework mudah tercampur:

1. **teks penalaran** yang ditulis model (*Thought*),
2. **aksi yang diusulkan** di teks itu (*Action*, *Action Input*),
3. **aksi yang berhasil diurai** program kita,
4. **eksekusi alat** yang sungguh terjadi, dan
5. **Observation** — hasil alat yang ditempel kembali ke prompt.

Model dihentikan di `\nObservation:` agar ia tidak mengarang hasil alatnya sendiri; bagian Observation selalu ditulis oleh program, bukan oleh model.

In [10]:
HAND_PROMPT = """Answer the question. You can use these tools:
Search: look up a city in the encyclopedia. Input: the city name.
Calculator: evaluate an arithmetic expression such as 3 * (4 + 5).

Use exactly this format:
Thought: what to do next
Action: Search or Calculator
Action Input: the input for the tool
Then wait for the Observation. Repeat Thought/Action/Action Input as needed, and finish with:
Final Answer: the answer

Question: {question}
"""
HAND_TOOLS = {"Search": search_city, "Calculator": calculator}


def react_by_hand(llm, question, max_steps=5):
    scratchpad = ""
    for step in range(1, max_steps + 1):
        text = llm.invoke(HAND_PROMPT.format(question=question) + scratchpad, stop=["\nObservation:"]).content.strip()
        print(f"[langkah {step}] teks model:\n  " + text.replace("\n", "\n  "))
        final = re.search(r"Final Answer:\s*(.*)", text)
        if final:
            print("  -> jawaban akhir:", final.group(1).strip())
            return final.group(1).strip()
        m = re.search(r"Action:\s*(\w+)\s*\nAction Input:\s*(.*)", text)
        if m is None or m.group(1) not in HAND_TOOLS:
            print("  -> tidak ada aksi yang bisa diurai; loop berhenti")
            return None
        tool, arg = m.group(1), m.group(2).strip()
        observation = HAND_TOOLS[tool](arg)
        print(f"  -> aksi terurai: {tool}({arg!r})\n  -> Observation dari alat: {observation}")
        scratchpad += f"{text}\nObservation: {observation}\n"
    print("  -> batas langkah tercapai")
    return None


Q_HAND = f"In which year was {c0['city']} founded, and how many years before 2026 was that?"
answer = react_by_hand(agent_llm(96), Q_HAND)
print("kunci:", c0["founded"], "dan", 2026 - int(c0["founded"]), "tahun")

[langkah 1] teks model:
  Thought: To find the year Ostmund was founded, I need to search for information about Ostmund. Then, I need to calculate how many years before 2026 that year was.
  Action: Search
  Action Input: Ostmund
  -> aksi terurai: Search('Ostmund')
  -> Observation dari alat: Ostmund is a city in Ostravia. Ostmund was founded in 1886. Ostmund lies on the river Kaskel. The population of Ostmund is 739,844. The mayor of Ostmund is Vera Varquist.
[langkah 2] teks model:
  Thought: To find out how many years before 2026 Ostmund was founded, I need to subtract the year Ostmund was founded from 2026.
  Action: Calculator
  Action Input: 2026 - 1886
  -> aksi terurai: Calculator('2026 - 1886')
  -> Observation dari alat: 140
[langkah 3] teks model:
  Final Answer: Ostmund was founded in 1886, which was 140 years before 2026.
  -> jawaban akhir: Ostmund was founded in 1886, which was 140 years before 2026.
kunci: 1886 dan 140 tahun


Di run validasi, Qwen menyelesaikan soal dalam tiga langkah persis seperti yang diharapkan: `Search('Ostmund')`, lalu `Calculator('2026 - 1886')`, lalu *Final Answer* yang menyebut 1886 dan 140 tahun.

Perhatikan pembagian kerjanya. Model hanya menulis teks. Angka 1886 datang dari Observation alat `Search`, dan 140 dari `Calculator`. Program kita yang mengurai baris *Action*, menjalankan alat, dan menulis Observation. Itulah seluruh mesin ReAct; LangChain mengotomatiskan hal yang sama dengan template, parser, dan executor.

## ReAct di LangChain: agen bawaan

Buku memakai kelas agen LangChain dalam tujuh langkah: memasang paket, menyiapkan model dan alat, membuat agen, memeriksa prompt-nya, mengganti prompt, mengganti deskripsi alat, dan menjalankan satu soal. Dua langkah pertama sudah tidak bisa dijalankan apa adanya.

> **Listing buku (langkah 1–2): paket dan import yang sudah pindah**
>
> ```python
> # !pip install langchain openai duckduckgo-search youtube_search wikipedia langchainhub
> os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter Your OpenAI API Key:")
> from langchain.agents import load_tools
> from langchain.chat_models import ChatOpenAI
> tools = load_tools(['wikipedia', 'ddg-search','llm-math'], llm=llm)
> ```
>
> Dengan `langchain` 1.4.6, kedua import gagal dengan `ImportError`: API agen lama sudah pindah ke paket lain yang tidak dipasang baris pip buku. Lokasi barunya:
>
> | Dipakai buku | Lokasi sekarang |
> |---|---|
> | `initialize_agent`, `AgentType`, `AgentExecutor`, `Tool` | `langchain_classic.agents` |
> | `DocstoreExplorer` | `langchain_classic.agents.react.base` |
> | `format_log_to_str`, `ReActSingleInputOutputParser` | `langchain_classic.agents.format_scratchpad`, `.output_parsers` |
> | `load_tools`, `Wikipedia`, `DuckDuckGoSearchRun` | `langchain_community` (`load_tools` juga butuh `mypy_extensions` yang tak dideklarasikan) |
> | `ChatOpenAI`, `OpenAI` | `langchain_openai` (butuh kunci API) |
>
> Notebook ini tidak memakai kunci API, Wikipedia, maupun DuckDuckGo di jalur utama; alatnya diganti `Search` dan `Calculator` lokal di atas.

> **Listing buku (langkah 3): `SyntaxError` dari spasi di nama enum**
>
> ```python
> agent=AgentType.ZERO_SHOT_REACT_ DESCRIPTION,
> ```
>
> Teks buku mencetak nama enum dengan spasi di tengah, sehingga Python membacanya sebagai dua nama berjajar dan seluruh sel gagal dengan `SyntaxError: invalid syntax. Perhaps you forgot a comma?` sebelum satu baris pun jalan. Nama yang benar adalah `ZERO_SHOT_REACT_DESCRIPTION`.

Sel berikut adalah langkah 3 yang bekerja. Tiga pengaturan ditambahkan di luar listing: `handle_parsing_errors=True`, supaya teks model yang tak berformat dikembalikan sebagai Observation alih-alih menghentikan seluruh run (bawaannya `False`); `max_iterations=8` sebagai rem (bawaannya 15); dan `return_intermediate_steps=True`, agar langkah-langkahnya bisa dibedah nanti. Sel ini juga mencetak `LangChainDeprecationWarning`: `initialize_agent` masih bekerja tetapi ditandai usang, dan LangChain menyarankan `langchain.agents.create_agent`. Notebook tetap memakainya karena itulah API di buku; peringatannya cukup tampil sekali.

In [11]:
from langchain_classic.agents import AgentType, initialize_agent
from langchain_core.tools import Tool

search_tool = Tool(name="Search", func=search_city, description="Look up a city in the local encyclopedia. Input should be the city name.")
calc_tool = Tool(name="Calculator", func=calculator, description="Evaluate an arithmetic expression such as 3 * (4 + 5).")
tools = [search_tool, calc_tool]
llm = agent_llm(128)
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,
                         handle_parsing_errors=True, max_iterations=8, return_intermediate_steps=True)
warnings.filterwarnings("ignore", message=".*create_agent.*")   # peringatan usang ini cukup tampil sekali
print(f"{type(agent).__name__} membungkus {type(agent.agent).__name__} | stop sequence: {agent.agent._stop}")

AgentExecutor membungkus ZeroShotAgent | stop sequence: ['\nObservation:', '\n\tObservation:']


/tmp/ipykernel_5544/4050968508.py:8: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True,


Langkah 4 buku: memeriksa prompt yang dirakit agen.

In [12]:
print(agent.agent.llm_chain.prompt.template)

Answer the following questions as best you can. You have access to the following tools:

Search(query) - Look up a city in the local encyclopedia. Input should be the city name.
Calculator(expression) - Evaluate an arithmetic expression such as 3 * (4 + 5).

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [Search, Calculator]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


Inilah template ReAct bawaan LangChain. Daftar alat dirakit otomatis dari objek `Tool` — termasuk tanda tangan fungsinya, `Search(query)` — lalu blok format *Question / Thought / Action / Action Input / Observation*, dan prompt berakhir di `Thought:` agar model langsung mulai berpikir. Prosa buku menyebut template agen seperti ini sering memuat contoh *few-shot*; template `ZERO_SHOT_REACT_DESCRIPTION` justru tidak memuat satu contoh pun — sesuai namanya, *zero-shot*. Model harus menyimpulkan format hanya dari instruksi.

Stop sequence yang dicetak saat agen dibuat, `\nObservation:` (beserta varian ber-tab `\n\tObservation:`), adalah pasangan dari template ini: model dihentikan tepat sebelum menulis hasil alat.

Langkah 5 buku menulis prompt kustom berisi prinsip ReAct, daftar alat, dan format interaksi. Listing-nya hanya membuat sebuah string, jadi bisa dijalankan apa adanya:

In [13]:
prompt = """
You are an intelligent agent designed to solve complex queries by breaking them down systematically and using available tools strategically. Follow the ReAct (Reasoning and Acting) framework to approach each task.
ReAct Principles:
1. Reasoning: Always start by carefully analyzing the question and developing a clear, step-by-step thought process.
2. Tool Selection: Critically evaluate which tools will be most effective for addressing the specific query.
3. Iterative Interaction: Be prepared to cycle between reasoning and action multiple times, refining your approach as you gather more information.
4. Comprehensive Understanding: Aim to not just find an answer, but to truly comprehend the underlying context and nuances of the question.
5. Transparent Decision-Making: Clearly articulate your reasoning, actions, and thought process at each step.
Available Tools:
- Wikipedia: Retrieve factual information about people, places, historical events, and general knowledge topics.
- Google Search: Fetch current information, recent events, and up-to-date context.
- Calculator: Perform mathematical calculations and numerical analysis.
Interaction Format:
Question: The specific query to be solved
Thought: Detailed reasoning about the approach, breaking down the problem
Action: Selected tool (Wikipedia/Google Search/Calculator)
Action Input: Precise query for the selected tool
Observation: Results obtained from the tool
... (Repeat reasoning, action, and observation as needed)
Thought: Final synthesized understanding
Final Answer: Comprehensive and well-reasoned response to the original question
Important Guidelines:
- Be methodical and explicit in your reasoning
- Use tools judiciously and avoid unnecessary actions
- Integrate information from multiple sources when appropriate
- Provide a clear, concise, and informative final answer
Begin!
Question: {input}
Thought:{agent_scratchpad}
"""

> **Listing buku (langkah 5): prompt kustom yang tidak pernah dipasang**
>
> Prosa buku mengatakan `agent.agent.llm_chain.prompt.template = prompt` memperbarui prompt agen, tetapi baris itu tidak ada di listing. Selain itu, prompt kustom menyebut alat `Wikipedia`, `Google Search`, dan `Calculator`, padahal alat yang dimuat buku bernama `wikipedia`, `duckduckgo_search`, dan `Calculator`, dan prompt itu tidak punya slot `{tools}` atau `{tool_names}`.

Sel berikut memeriksa kedua hal itu. Untuk menguji mesin agennya saja, kita memakai `FakeListLLM` — **naskah tetap, bukan model bahasa** — yang menulis aksi persis menurut prompt kustom.

In [14]:
from langchain_core.language_models.fake import FakeListLLM

print("template agen sama dengan prompt kustom:", agent.agent.llm_chain.prompt.template == prompt)
named = re.findall(r"^- ([A-Z][\w ]*):", prompt, re.M)
print("alat yang disebut prompt kustom:", named, "| alat yang terdaftar:", [t.name for t in tools])

# Naskah tetap (FakeListLLM, bukan model bahasa) yang patuh pada prompt kustom: yang diuji hanya mesin agennya
script = [f"Thought: I should look this up.\nAction: Wikipedia\nAction Input: {c0['city']}",
          "Thought: Let me compute.\nAction: Calculator\nAction Input: 2 * 21",
          "Thought: Final synthesized understanding\nFinal Answer: (naskah selesai)"]
scripted = initialize_agent(tools, FakeListLLM(responses=script), agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                            return_intermediate_steps=True)
scripted.agent.llm_chain.prompt.template = prompt   # baris dari prosa buku yang tidak ada di listingnya
result = scripted.invoke({"input": f"What is the population of {c0['city']}?"})
for action, observation in result["intermediate_steps"]:
    print(f"   Action {action.tool!r} -> Observation: {observation}")

template agen sama dengan prompt kustom: False
alat yang disebut prompt kustom: ['Wikipedia', 'Google Search', 'Calculator'] | alat yang terdaftar: ['Search', 'Calculator']
   Action 'Wikipedia' -> Observation: Wikipedia is not a valid tool, try one of [Search, Calculator].
   Action 'Calculator' -> Observation: 42


Template agen tetap bawaan (`False`), jadi listing buku tidak mengubah apa pun. Setelah baris dari prosa itu dipasang ke agen kedua, mesinnya bekerja, tetapi aksi `Wikipedia` dijawab *is not a valid tool*: `AgentExecutor` mencocokkan nama alat persis, huruf demi huruf. Hanya `Calculator` yang lolos, dan hasilnya `42` berasal dari eksekusi kalkulator sungguhan. Pelajarannya sederhana: nama alat di prompt harus sama persis dengan nama objek `Tool`, dan cara paling aman adalah membiarkan LangChain mengisinya lewat `{tool_names}`.

> **Listing buku (langkah 6): deskripsi alat diubah setelah agen dibuat**
>
> ```python
> tools[1].description = "A date retrieval tool that provides the current date and time, ..."
> tools[2].description = "A powerful computational tool ... unit conversions, and advanced mathematical functions."
> ```
>
> Di buku, `tools[1]` adalah alat pencarian DuckDuckGo, jadi menyebutnya "pengambil tanggal" keliru. `tools[2]` adalah `llm-math`, yang evaluatornya hanya menghitung ekspresi angka. Masalah ketiga ada pada urutan: agen sudah dibuat di langkah 3.

Kita ulang pengubahan deskripsi kalkulator itu ke agen yang sudah ada, lalu bandingkan dengan agen yang dibangun ulang.

In [15]:
calc_description = calc_tool.description
calc_tool.description = ("A powerful computational tool capable of performing various mathematical operations, including arithmetic "
                         "calculations, algebraic computations, percentage calculations, unit conversions, and advanced mathematical functions.")
print("deskripsi baru ada di prompt agen yang sudah dibuat:", "unit conversions" in agent.agent.llm_chain.prompt.template)
rebuilt = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
print("deskripsi baru ada setelah agen dibangun ulang   :", "unit conversions" in rebuilt.agent.llm_chain.prompt.template)
for expr in ["37593 ** (1/5)", "0.15 * 240", "x**2 - 4", "5 km in miles"]:
    print(f"   calculator({expr!r}) -> {calculator(expr)}")
calc_tool.description = calc_description   # kembalikan deskripsi yang jujur untuk sisa notebook

deskripsi baru ada di prompt agen yang sudah dibuat: False
deskripsi baru ada setelah agen dibangun ulang   : True
   calculator('37593 ** (1/5)') -> 8.22283
   calculator('0.15 * 240') -> 36
   calculator('x**2 - 4') -> Error: only numbers and + - * / ( ) are allowed
   calculator('5 km in miles') -> Error: only numbers and + - * / ( ) are allowed


`initialize_agent` merangkai deskripsi alat menjadi string template saat agen dibuat. Mengubah `description` sesudahnya tidak menyentuh string itu (`False`); deskripsi pengganti itu baru terbaca setelah agen dibangun ulang (`True`). Isi deskripsi juga perlu jujur: kalkulator menghitung `37593 ** (1/5)` dan `0.15 * 240`, tetapi menolak aljabar (`x**2 - 4`) dan konversi satuan (`5 km in miles`). Deskripsi yang menjanjikan kemampuan lebih hanya membuat model mengirim input yang pasti gagal. Deskripsi aslinya dikembalikan untuk sisa notebook.

Langkah 7: menjalankan agen. Soal buku menanyakan populasi kota terbesar Kanada, waktu yang dibutuhkan penduduknya untuk menghitung sampai satu miliar, dan perbandingannya dengan umur manusia — butuh pencarian web. Versi di sini mempertahankan bentuk soal multi-langkah itu di kota fiktif: cari populasi, lalu hitung. Jawaban yang benar butuh **dua** alat: `Search` untuk populasi dan `Calculator` untuk 10⁹ ÷ populasi ÷ 86.400 detik per hari.

`verbose=True` mencetak jejaknya: teks hijau ditulis model, dan Observation dari alat diberi warna per alat (biru muda/sian untuk `Search`, kuning untuk `Calculator`).

In [16]:
Q_COUNT = (f"What is the population of {c0['city']}? How many days would it take for that city's population to count to "
           "1 billion if each person counts one number per second without breaks?")
TOOL_LOG.clear()
result = agent.invoke({"input": Q_COUNT})



> Entering new AgentExecutor chain...
I need to find the population of Ostmund and then calculate how many seconds it would take for the population to reach 1 billion.
Action: Search
Action Input: Ostmund
Observation: Ostmund is a city in Ostravia. Ostmund was founded in 1886. Ostmund lies on the river Kaskel. The population of Ostmund is 739,844. The mayor of Ostmund is Vera Varquist.
Thought:I now know the final answer
Final Answer: The population of Ostmund is 739,844. To reach 1 billion, it would take 1,317,000 seconds, which is approximately 33 years and 11 months.

> Finished chain.


Jejak verbose bercampur warna. Sel berikut memisahkan setiap langkah menjadi teks model, aksi yang terurai, dan Observation, lalu membandingkan jawaban dengan hitungan yang benar.

In [17]:
pop = int(c0["population"].replace(",", ""))
truth = 1e9 / pop / 86400
for i, (action, observation) in enumerate(result["intermediate_steps"], 1):
    print(f"langkah {i}")
    print("   teks model    :", action.log.strip().replace("\n", " | "))
    print("   aksi terurai  :", f"{action.tool}({action.tool_input!r})")
    print("   Observation   :", observation)
print("jawaban akhir   :", result["output"])
print(f"kunci           : 1e9 / {pop:,} / 86400 = {truth:.5f} hari (= {truth * 24 * 60:.1f} menit)")
print("eksekusi alat   :", TOOL_LOG)

langkah 1
   teks model    : I need to find the population of Ostmund and then calculate how many seconds it would take for the population to reach 1 billion. | Action: Search | Action Input: Ostmund
   aksi terurai  : Search('Ostmund')
   Observation   : Ostmund is a city in Ostravia. Ostmund was founded in 1886. Ostmund lies on the river Kaskel. The population of Ostmund is 739,844. The mayor of Ostmund is Vera Varquist.
jawaban akhir   : The population of Ostmund is 739,844. To reach 1 billion, it would take 1,317,000 seconds, which is approximately 33 years and 11 months.
kunci           : 1e9 / 739,844 / 86400 = 0.01564 hari (= 22.5 menit)
eksekusi alat   : [('Search', 'Ostmund')]


Di run validasi, jejaknya memperlihatkan separuh ReAct yang berhasil dan separuh yang gagal. Langkah pertama benar: model mencari Ostmund dan Observation memuat populasi 739,844. Tetapi di langkah kedua model langsung menulis *Final Answer* tanpa memanggil `Calculator`, dan hitungannya sendiri salah: "1,317,000 detik, sekitar 33 tahun 11 bulan". Yang benar: satu miliar dibagi 739.844 penduduk ≈ 1.352 detik — sekitar 22,5 menit (0,0156 hari), karena seluruh penduduk menghitung bersamaan. Dua angka di jawaban model bahkan tidak saling cocok: 1,3 juta detik kira-kira 15 hari, bukan 33 tahun. `TOOL_LOG` mengonfirmasi hanya satu alat yang benar-benar dijalankan.

*Thought*-nya sudah memperlihatkan salah tangkap: model berniat menghitung kapan *populasinya* mencapai satu miliar, bukan berapa lama penduduknya menghitung. Pelajarannya: ReAct menyediakan kalkulator, tetapi model yang memutuskan kapan memakainya. Di contoh tangan tadi Qwen memakai `Calculator` untuk pengurangan sederhana; di soal yang lebih panjang ini ia memilih menghitung di kepala. Untuk tugas yang menuntut angka tepat, agen perlu diperiksa di level aksi, bukan hanya dari jawaban akhirnya.

Bagaimana kalau otak agennya model dasar? Sel berikut menjalankan agen yang sama dengan GPT-2, dengan `max_iterations` bawaan 15. Teks GPT-2 yang tak berformat dikembalikan sebagai Observation, dan prompt bertambah panjang di setiap iterasi.

In [18]:
from langchain_core.exceptions import OutputParserException

gpt2_agent = initialize_agent(tools, base_llm(64), agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                              handle_parsing_errors=True, return_intermediate_steps=True)   # max_iterations bawaan: 15
n0 = len(CALLS)
try:
    outcome = gpt2_agent.invoke({"input": Q_COUNT})["output"]
except ContextOverflow as err:   # prompt yang terus tumbuh akhirnya melewati konteks 1.024 token GPT-2
    outcome = f"dihentikan penjaga konteks ({err})"
calls = [c for c in CALLS[n0:] if not c["overflow"]]
valid = 0
for c in calls:
    try:
        step = gpt2_agent.agent.output_parser.parse(c["text"])
        valid += getattr(step, "tool", None) in {"Search", "Calculator"}
    except OutputParserException:   # teks yang tak terurai dihitung sebagai bukan aksi
        pass
print(f"panggilan GPT-2: {len(calls)} | aksi ke alat yang sah: {valid} | token prompt: {calls[0]['prompt_tokens']} -> {calls[-1]['prompt_tokens']}")
print("hasil:", outcome)
print("teks GPT-2 di panggilan pertama:", repr(calls[0]["text"][:160]))

panggilan GPT-2: 12 | aksi ke alat yang sah: 0 | token prompt: 213 -> 950
hasil: dihentikan penjaga konteks (GPT-2: prompt 1017 token + 64 token baru > konteks 1024)
teks GPT-2 di panggilan pertama: ' 1 billion\n\nAction: the action to take, should be one of [Search, Calculator]\n\nAction Input: the input to the action\n'


GPT-2 tidak pernah sampai ke alat. Di panggilan pertama ia menyalin baris format template apa adanya — *Action: the action to take, should be one of [Search, Calculator]* dan *Action Input: the input to the action* — sehingga parser menerima kalimat placeholder itu sebagai nama alat, dan executor membalas bahwa alat itu tidak ada. Di run validasi, 12 panggilan menghasilkan 0 aksi yang sah.

Setiap iterasi menempelkan teks GPT-2 dan pesan galat ke prompt, sehingga prompt tumbuh dari 213 ke 950 token. Panggilan ke-13 butuh 1.017 + 64 token, melebihi konteks GPT-2 (1.024), dan penjaga konteks menghentikan run sebelum batas 15 iterasi tercapai. Prosa buku menggambarkan loop yang berulang sampai agen memutuskan informasinya cukup; dalam praktik ada jalan keluar lain: exception parser (bila `handle_parsing_errors=False`, nilai bawaannya), batas iterasi, dan — untuk model kecil — batas konteks.

Dua model ini berbeda ukuran *dan* pelatihan, jadi perbandingan ini tidak memisahkan keduanya. Tetapi menyalin placeholder adalah ciri model dasar yang sekadar melanjutkan teks; eksperimen *Under the hood* pertama nanti (UTH-1) memperlihatkan bahwa GPT-2 bisa menulis aksi yang benar begitu diberi contoh.

## ReAct di atas *document store*

Agen tipe `REACT_DOCSTORE` memakai dua alat: `Search` mengambil sebuah dokumen dan mengembalikan paragraf pertamanya, lalu `Lookup` mencari paragraf yang memuat kata kunci tertentu di dokumen itu. Pola ini cocok untuk dokumen panjang: model tidak perlu membaca semuanya, cukup paragraf yang relevan.

> **Listing buku: import yang hilang**
>
> ```python
> from langchain import Wikipedia
> from langchain.agents.react.base import DocstoreExplorer
> llm = OpenAI(temperature=0)
> ```
>
> Baris pertama gagal dengan `ImportError` dan baris kedua dengan `ModuleNotFoundError` (lihat tabel lokasi impor di atas). Listing juga tidak pernah mengimpor `OpenAI`, sehingga `llm = OpenAI(temperature=0)` akan berhenti dengan `NameError`. Deskripsi alat yang ditulis listing pun tidak pernah sampai ke model, seperti diperiksa di bawah.

Di sini `DocstoreExplorer` buku dipasang di atas ensiklopedia lokal, dengan dokumen berparagraf: paragraf pertama berisi negara, tahun, dan sungai; populasi dan wali kota ada di paragraf terpisah. Jadi soal populasi menuntut `Search` lalu `Lookup`.

Seperti agen bawaan tadi, `handle_parsing_errors=True` dan batas iterasi (`max_iterations=6`) ditambahkan di luar listing. `Lookup` juga dibungkus agar galat "Lookup sebelum Search berhasil" kembali ke model sebagai Observation.

In [19]:
from langchain_classic.agents.react.base import DocstoreExplorer
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document


def safe_lookup(explorer):
    def lookup(term):
        try:
            return explorer.lookup(term)
        except ValueError as err:   # Lookup sebelum Search berhasil: dikembalikan ke model sebagai Observation
            return f"Error: {err}"
    return lookup


docstore = DocstoreExplorer(InMemoryDocstore({c["city"]: Document(page_content=paragraphs(c)) for c in CITIES}))
ds_tools = [Tool(name="Search", func=docstore.search, description="Search for latest information about any topic"),
            Tool(name="Lookup", func=safe_lookup(docstore), description="Lookup tool for get information from a keyword")]
react = initialize_agent(ds_tools, agent_llm(128), agent=AgentType.REACT_DOCSTORE, verbose=True,
                         handle_parsing_errors=True, max_iterations=6)
c1 = KB[1]
Q_DS = f"What is the population of {c1['city']}?"
p0 = react.agent.llm_chain.prompt.format(input=Q_DS, agent_scratchpad="")
print(f"prompt REACT_DOCSTORE: {len(base_tok(p0)['input_ids']):,} token GPT-2 (konteks GPT-2 {base_model.config.n_positions:,}) | "
      f"{len(agent_tok(p0)['input_ids']):,} token {AGENT_LABEL}")
print("deskripsi alat dari listing muncul di prompt:", [t.description in p0 for t in ds_tools])
print("Search:", docstore.search(c1["city"]))
print("Lookup:", safe_lookup(docstore)("population"))

prompt REACT_DOCSTORE: 1,430 token GPT-2 (konteks GPT-2 1,024) | 1,418 token Qwen2.5-1.5B-Instruct
deskripsi alat dari listing muncul di prompt: [False, False]
Search: Lumstad is a city in Ostravia. Lumstad was founded in 1412. Lumstad lies on the river Lennor.
Lookup: (Result 1/1) The population of Lumstad is 859,888.


Dua hal terlihat sebelum model dipanggil. Pertama, prompt `REACT_DOCSTORE` membawa enam contoh lengkap dari Wikipedia, sehingga panjangnya 1.430 token GPT-2 — lebih panjang dari seluruh konteks GPT-2 (1.024). Agen ini mustahil dijalankan dengan GPT-2; untuk Qwen masih muat di batas 4.096 token yang kita izinkan.

Kedua, deskripsi alat dari listing buku tidak muncul di prompt (`[False, False]`). Template ini tetap, tanpa slot deskripsi, jadi model mengenal `Search` dan `Lookup` hanya dari contoh. Dua baris terakhir memperlihatkan mesinnya: `Search` mengembalikan paragraf pertama dokumen Lumstad (tanpa populasi), dan `Lookup[population]` mengembalikan paragraf yang memuat 859,888.

In [20]:
out = react.invoke({"input": Q_DS})["output"]
print("jawaban:", out, "| kunci:", c1["population"])



> Entering new AgentExecutor chain...
Thought: I need to search for the population of Lumstad. I will look for the population of the city of Lumstad, which is a small town in Minnesota.
Action: Search[Lumstad]
Observation: Lumstad is a city in Ostravia. Lumstad was founded in 1412. Lumstad lies on the river Lennor.
Thought:I need to search for the population of Lumstad, which is a small town in Minnesota. I will look for the population of the city of Lumstad, which is a small town in Minnesota.
Action: Search[Lumstad population]
Observation: ID Lumstad population not found.
Thought:I need to search for the population of Lumstad, which is a small town in Minnesota. I will look for the population of the city of Lumstad, which is a small town in Minnesota.
Action: Search[Lumstad population]
Observation: ID Lumstad population not found.
Thought:Could not parse LLM Output: I apologize for the confusion, but the information provided does not contain the specific population data for Lumstad

Di run validasi agen ini gagal, dan cara gagalnya instruktif. Sejak *Thought* pertama — sebelum ada hasil alat apa pun — model sudah mengarang bahwa Lumstad adalah "kota kecil di Minnesota". Aksinya tetap benar (`Search[Lumstad]`), dan Observation menyebut Ostravia, tetapi *Thought* berikutnya mengulang Minnesota. Setelah paragraf pertama tidak memuat populasi, model tidak memakai `Lookup`; ia dua kali menulis `Search[Lumstad population]` (*not found*), lalu menulis permintaan maaf tanpa format — yang menggabungkan keduanya menjadi "Ostravia, Minnesota" — sampai batas 6 iterasi habis. Padahal jawabannya cukup didapat dengan satu `Lookup[population]`.

Dua pelajaran. Contoh yang panjang belum tentu dipahami model kecil: pola Search→Lookup hanya muncul di dua dari enam contoh (tiga contoh lain memakai dua `Search` untuk dua entitas berbeda, dan satu mengulang `Search` dengan judul yang lebih tepat setelah *Could not find*), dan model tidak memakainya. Selain itu, *Thought* tidak menjamin penalaran yang berpijak pada Observation — model bisa mempertahankan keyakinan yang bertentangan dengan Observation di depannya.

**Opsional — contoh Wikipedia dari buku.** Dengan `RUN_WIKIPEDIA = True`, sel berikut memasang paket `wikipedia` dan menjalankan soal buku (*Who is the current governor of Texas and when was he born?*) dengan `DocstoreExplorer(Wikipedia())`. Alatnya hanya membaca, tetapi butuh internet dan hasilnya mengikuti isi Wikipedia saat dijalankan. Sel ini tidak dijalankan pada run validasi. Halaman Wikipedia yang panjang bisa memenuhi konteks; penjaga konteks lalu menghentikan contoh ini dengan pesan yang jelas. Kalau jaringan atau Wikipedia bermasalah, sel ini berhenti dengan error: set `RUN_WIKIPEDIA = False` atau lanjutkan dari sel berikutnya.

In [21]:
if RUN_WIKIPEDIA:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wikipedia==1.4.0"], check=True)
    from langchain_community.docstore.wikipedia import Wikipedia
    wiki = DocstoreExplorer(Wikipedia())
    wiki_tools = [Tool(name="Search", func=wiki.search, description="Search Wikipedia"),
                  Tool(name="Lookup", func=safe_lookup(wiki), description="Lookup a keyword in the current page")]
    react_wiki = initialize_agent(wiki_tools, agent_llm(128), agent=AgentType.REACT_DOCSTORE, verbose=True,
                                  handle_parsing_errors=True, max_iterations=6)
    try:
        print(react_wiki.invoke({"input": "Who is the current governor of Texas and when was he born?"})["output"])
    except ContextOverflow as err:   # halaman Wikipedia panjang bisa memenuhi konteks 4.096 token
        print(f"dihentikan penjaga konteks ({err})")
else:
    print("Dilewati: RUN_WIKIPEDIA = False.")

Dilewati: RUN_WIKIPEDIA = False.


## Agen ReAct dengan LangChain Expression Language

LCEL menyusun agen sebagai pipeline yang bisa dibaca dari kiri ke kanan: masukan dan *scratchpad* (riwayat langkah sebelumnya yang diformat `format_log_to_str`) → prompt → model yang dihentikan di `\nObservation:` → `ReActSingleInputOutputParser`, yang mengubah teks model menjadi `AgentAction` (alat + input) atau `AgentFinish` (jawaban akhir).

> **Listing buku: import LCEL yang sudah pindah**
>
> ```python
> from langchain.agents.format_scratchpad import format_log_to_str
> from langchain.agents.output_parsers import ReActSingleInputOutputParser
> from langchain.tools import DuckDuckGoSearchRun
> ```
>
> Dua baris pertama gagal dengan `ModuleNotFoundError`, baris ketiga dengan `ImportError`. Lokasi barunya ada di tabel di atas.

Selain import, model, dan alatnya (`Search` dan `Calculator` lokal menggantikan `DuckDuckGoSearchRun`), sel berikut mengikuti listing buku.

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents.format_scratchpad import format_log_to_str              # dulu: langchain.agents.format_scratchpad
from langchain_classic.agents.output_parsers import ReActSingleInputOutputParser      # dulu: langchain.agents.output_parsers

# 1. Define Tools: alat lokal yang terkendali menggantikan DuckDuckGoSearchRun
lcel_tools = [search_tool, calc_tool]
# 2. Construct the Prompt: Instead of pulling from a hub, we'll define a basic prompt template.
template = """Answer the following questions as best you can. You have access to the following tools:
{tool_descriptions}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
{agent_scratchpad}"""
prompt = ChatPromptTemplate.from_template(template)
prompt = prompt.partial(
    tool_names=", ".join([t.name for t in lcel_tools]),
    tool_descriptions="\n".join(
        [f"{t.name}: {t.description}" for t in lcel_tools]
    ),
)
# 3. Instantiate the LLM: model lokal di GPU menggantikan ChatOpenAI
llm = agent_llm(128)
# We also configure it to stop when it sees '\nObservation:'
llm_with_stop = llm.bind(stop=["\nObservation:"])
# 4. Construct the Agent Pipeline using LCEL:
lcel_agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x:
            format_log_to_str(x["intermediate_steps"]),
    }
    | prompt
    | llm_with_stop
    | ReActSingleInputOutputParser()
)
print(type(lcel_agent).__name__)

RunnableSequence


Pipeline tadi hanya menghasilkan *satu* langkah. `AgentExecutor` yang menjalankan loop: memanggil pipeline, mengeksekusi alat yang dipilih, menambahkan hasilnya ke *scratchpad*, dan mengulang sampai muncul `AgentFinish`. Import `from langchain.agents import AgentExecutor` di buku juga gagal (`ImportError`); kelasnya kini di `langchain_classic.agents`.

Soal buku (CEO Microsoft dan kuadrat umurnya) butuh pencarian web, jadi kita pakai padanan lokal: wali kota sebuah kota fiktif dan kuadrat tahun berdirinya — satu pencarian, satu hitungan. Di luar listing, executor diberi `handle_parsing_errors=True` dan `max_iterations=8` seperti agen bawaan, serta `return_intermediate_steps=True` agar aksinya bisa dicetak.

In [23]:
from langchain_classic.agents import AgentExecutor   # dulu: langchain.agents

c2 = KB[2]
Q_LCEL = f"Who is the mayor of {c2['city']}, and what is the city's founding year squared?"
agent_executor = AgentExecutor(agent=lcel_agent, tools=lcel_tools, verbose=True, handle_parsing_errors=True,
                               max_iterations=8, return_intermediate_steps=True)
response = agent_executor.invoke({"input": Q_LCEL})
print("\njawaban:", response["output"])
print("kunci  :", c2["mayor"], "|", int(c2["founded"]) ** 2, f"({c2['founded']}^2)")
print("aksi   :", [(a.tool, a.tool_input) for a, _ in response["intermediate_steps"]])



> Entering new AgentExecutor chain...
Thought: I need to find the current mayor of Rinvale and the founding year of the city. I should use the Search tool to find this information.
Action: Search
Action Input: RinvaleRinvale is a city in Thalmund. Rinvale was founded in 1483. Rinvale lies on the river Ambrel. The population of Rinvale is 207,939. The mayor of Rinvale is Oskar Praskov.I now know the final answer
Final Answer: The mayor of Rinvale is Oskar Praskov, and the city's founding year squared is 1483^2 = 2202649.

> Finished chain.

jawaban: The mayor of Rinvale is Oskar Praskov, and the city's founding year squared is 1483^2 = 2202649.
kunci  : Oskar Praskov | 2199289 (1483^2)
aksi   : [('Search', 'Rinvale')]


Pipeline LCEL dan `AgentExecutor` bekerja: satu `Search`, Observation berisi wali kota (Oskar Praskov) dan tahun berdiri (1483), lalu *Final Answer*. Wali kotanya benar. Kuadratnya tidak: di run validasi model menulis 1483² = 2.202.649, padahal hasilnya 2.199.289 — meleset 3.360, sekitar 0,15%: angka yang tampak masuk akal tetapi salah. Seperti pada agen bawaan, `Calculator` tersedia tetapi tidak dipakai. Di jejak verbose LCEL, hasil alat dicetak langsung setelah input aksi tanpa label *Observation:*; isinya tetap keluaran `Search`.

## Under the hood: membedah satu iterasi ReAct

Bagian *Under the hood* (UTH-1 sampai UTH-4) adalah eksperimen tambahan di luar buku. Demo di atas masing-masing satu soal. Untuk melihat pola, kita ulangi satu iterasi ReAct pada 30 soal ensiklopedia (10 populasi, 10 tahun berdiri, 10 wali kota), dengan komponen listing LCEL buku sendiri: template, `format_log_to_str`, dan `ReActSingleInputOutputParser`.

Dua template dibandingkan. **Buku (zero-shot)** adalah template LCEL apa adanya. **Few-shot** adalah template yang sama dengan tiga contoh jejak lengkap — satu per jenis pertanyaan — disisipkan sebelum `Begin!`.

Satu alat bantu lagi: **langkah oracle**. Untuk menguji langkah jawaban secara terpisah, aksi pertama (`Search` ke kota yang benar) ditulis tangan, Observation-nya diambil dari ensiklopedia, dan model hanya menulis langkah berikutnya. Dengan begitu kegagalan di langkah aksi tidak mengaburkan pengukuran langkah jawaban.

In [24]:
from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.exceptions import OutputParserException

ITEMS = [(c, TYPES[i % 3]) for i, c in enumerate(KB[:N_KB])]   # soal uji: (kota, jenis pertanyaan)
SHOT_TYPES = ["population", "mayor", "founded"]                 # tiga contoh few-shot, satu per jenis
STOP_BOOK = ["\nObservation:"]
STOP_PLUS = ["\nObservation:", "\nQuestion:"]
LLMS = {"GPT-2": base_llm, AGENT_LABEL: agent_llm}


def ex_trace(c, t):
    return (f"Question: {QTXT[t].format(c['city'])}\nThought: I need to look up {c['city']}.\n"
            f"Action: Search\nAction Input: {c['city']}\nObservation: {flat(c)}\n"
            f"Thought: I now know the final answer\nFinal Answer: {c[t]}\n\n")


def make_prompt(text, tool):
    return ChatPromptTemplate.from_template(text).partial(tool_names=tool.name, tool_descriptions=f"{tool.name}: {tool.description}")


def esc(s):
    return s.replace("{", "{{").replace("}", "}}")


HEAD = template.split("Begin!")[0]   # instruksi dan blok format dari template LCEL buku, apa adanya
EXAMPLES = "".join(ex_trace(c, t) for c, t in zip(SHOTS, SHOT_TYPES))
TPL = {"buku (zero-shot)": make_prompt(template, search_tool),
       "few-shot": make_prompt(HEAD + esc(EXAMPLES) + "Begin!\nQuestion: {input}\n{agent_scratchpad}", search_tool)}


def step_text(llm, tpl, question, steps, stop=None):
    messages = TPL[tpl].invoke({"input": question, "agent_scratchpad": format_log_to_str(steps)}).to_messages()
    return llm.invoke(messages, stop=stop).content


PARSER = ReActSingleInputOutputParser()


def parse(text):
    try:
        return PARSER.parse(text)
    except OutputParserException as err:
        return err


def oracle_action(c):   # langkah pertama yang benar, ditulis tangan (bukan oleh model)
    return AgentAction("Search", c["city"], f"Thought: I need to look up {c['city']}.\nAction: Search\nAction Input: {c['city']}")


def norm(s):
    return re.sub(r"[^0-9a-z ]", "", s.lower().replace(",", ""))


def mentions(answer, value):
    return f" {norm(value)} " in f" {norm(answer)} "


def ci(v, z=1.96):   # interval kepercayaan 95% Wilson untuk proporsi
    v = np.asarray(v, float)
    n = len(v)
    p = v.mean()
    center = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return f"[{max(0.0, center - half):.2f}; {min(1.0, center + half):.2f}]"


c, t = ITEMS[0]
for name in TPL:
    text = TPL[name].invoke({"input": QTXT[t].format(c["city"]), "agent_scratchpad": ""}).to_messages()[0].content
    print(f"{name:<17} prompt langkah 1: {len(base_tok(text)['input_ids'])} token GPT-2")

buku (zero-shot)  prompt langkah 1: 151 token GPT-2
few-shot          prompt langkah 1: 487 token GPT-2


Contoh few-shot membuat prompt tiga kali lebih panjang: sekitar 150 token tanpa contoh, hampir 490 token dengan tiga contoh. Biaya ini dibayar di setiap panggilan model.

### UTH-1 — Apakah model menulis aksi yang bisa dieksekusi?

Seluruh loop bergantung pada parser: teks yang tak terurai tidak pernah menjadi aksi. Untuk setiap soal, model menulis langkah pertama (64 token, tanpa stop), lalu teksnya dipotong di stop buku `\nObservation:` dan diurai. Kita hitung berapa yang menjadi aksi, berapa yang memanggil `Search`, dan apakah inputnya tepat nama kota — persis, dan setelah spasi/baris baru di tepi dipangkas. Kurung siku di kolom terakhir adalah interval kepercayaan 95% (metode Wilson): rentang tingkat keberhasilan yang masih cocok dengan hasil 30 soal. Misalnya 30 dari 30 memberi [0,89; 1,00] — data ini belum bisa membedakan model yang selalu berhasil dari model yang berhasil di sekitar 90% soal sejenis.

In [25]:
t0 = time.perf_counter()
S1 = {}   # teks langkah pertama tanpa stop sequence, per (model, template)
for m_name, make in LLMS.items():
    llm = make(64)
    for tpl in TPL:
        S1[m_name, tpl] = [step_text(llm, tpl, QTXT[t].format(c["city"]), []) for c, t in ITEMS]
print(f"{len(S1)} kombinasi x {N_KB} soal, {time.perf_counter() - t0:.0f} s; teks dipotong di stop buku sebelum diurai\n")
print(f"{'model':<22} {'template':<17} {'aksi terurai':>12} {'alat sah':>9} {'input persis':>12} {'input = kota*':>13}   CI95 (*)")
for (m_name, tpl), texts in S1.items():
    parsed = [parse(cut(x, STOP_BOOK)) for x in texts]
    act = [isinstance(r, AgentAction) for r in parsed]
    valid = [a and r.tool == "Search" for a, r in zip(act, parsed)]
    exact = [a and r.tool_input == c["city"] for a, r, (c, _) in zip(act, parsed, ITEMS)]
    stripped = [a and r.tool_input.strip() == c["city"] for a, r, (c, _) in zip(act, parsed, ITEMS)]
    print(f"{m_name:<22} {tpl:<17} {sum(act):>9}/{N_KB} {sum(valid):>6}/{N_KB} {sum(exact):>9}/{N_KB} {sum(stripped):>10}/{N_KB}   {ci(stripped)}")
print("(*) setelah spasi dan baris baru di tepi input dipangkas\n")
for (m_name, tpl), texts in S1.items():
    tools_seen = Counter(r.tool[:40] for r in (parse(cut(x, STOP_BOOK)) for x in texts) if isinstance(r, AgentAction))
    print(f"{m_name}, {tpl}: nama alat yang terurai {dict(tools_seen.most_common(3))}")
print()
for key, texts in S1.items():
    print(f"{key[0]}, {key[1]}, soal 1: {cut(texts[0], STOP_BOOK)[:110]!r}")

4 kombinasi x 30 soal, 180 s; teks dipotong di stop buku sebelum diurai

model                  template          aksi terurai  alat sah input persis input = kota*   CI95 (*)
GPT-2                  buku (zero-shot)         12/30      0/30         0/30          0/30   [0.00; 0.11]
GPT-2                  few-shot                 30/30     30/30         2/30         30/30   [0.89; 1.00]
Qwen2.5-1.5B-Instruct  buku (zero-shot)         30/30     30/30        30/30         30/30   [0.89; 1.00]
Qwen2.5-1.5B-Instruct  few-shot                 30/30     30/30        30/30         30/30   [0.89; 1.00]
(*) setelah spasi dan baris baru di tepi input dipangkas

GPT-2, buku (zero-shot): nama alat yang terurai {'the action to take, should be one of [Se': 11, 'the action to take, should be one of [Dr': 1}
GPT-2, few-shot: nama alat yang terurai {'Search': 30}
Qwen2.5-1.5B-Instruct, buku (zero-shot): nama alat yang terurai {'Search': 30}
Qwen2.5-1.5B-Instruct, few-shot: nama alat yang terurai {'Search'

Tabel ini memuat perbedaan paling tajam di notebook ini.

- **GPT-2 tanpa contoh** tidak pernah memanggil alat yang sah: 0 dari 30 (CI95 [0,00; 0,11], jadi tingkat sebenarnya kemungkinan besar tidak lebih dari sekitar 11%). Sebagian teksnya terurai sebagai aksi (12 dari 30), tetapi "alat"-nya selalu kalimat placeholder template yang disalin, *the action to take, should be one of [...]* (12 dari 12). Contoh soal 1 malah mengarang fakta tentang kota fiktif (*Ostmund is a city in the north of the continent*) — halusinasi yang hendak dicegah ReAct.
- **GPT-2 dengan tiga contoh** berubah total: 30 dari 30 menjadi `Search` ke kota yang benar. Bagi model dasar, format datang dari contoh, bukan dari instruksi.
- **Qwen** sudah 30 dari 30 tanpa contoh: baginya instruksi template sudah cukup. Ingat bahwa Qwen model instruksi *dan* sekitar 12 kali lebih besar, jadi tabel ini tidak memisahkan kedua faktor itu.

Ada satu angka kecil yang menentukan: input GPT-2 few-shot yang *persis* sama dengan nama kota hanya 2 dari 30. Sisanya berakhir dengan baris baru. GPT-2 menulis baris kosong sebelum *Observation*; stop `\nObservation:` hanya memotong baris baru yang kedua, dan parser tidak membuang yang pertama (`'Ostmund\n'`). Alat yang mencocokkan nama secara persis akan menjawab *not found* untuk 28 soal itu. Karena itulah `Search` di notebook ini memangkas spasi di tepi input — pertahanan murah di sisi alat.

### UTH-2 — Apa yang dicegah stop sequence?

Buku menghentikan model di `\nObservation:`. Teks langkah pertama template few-shot dari UTH-1 dibuat *tanpa* stop, jadi kita bisa melihat apa yang ditulis model bila tidak dihentikan: apakah ia mengarang Observation sendiri, dan apakah fakta di dalamnya cocok dengan ensiklopedia.

Bagian kedua menguji langkah jawaban: setelah aksi oracle dan Observation yang benar, model menulis 48 token tanpa stop, lalu teksnya dipotong dengan stop buku saja, atau stop buku ditambah `\nQuestion:` — baris yang membuka soal baru di contoh few-shot.

In [26]:
PAT = {"country": r"is a city in ([A-Z]\w+)", "founded": r"founded in (\d{3,4})", "river": r"on the river ([A-Z]\w+)",
       "population": r"population of \w+ is ([\d,]+\d)", "mayor": r"mayor of \w+ is ([A-Z]\w+ [A-Z]\w+)"}
print("(a) langkah aksi tanpa stop sequence (template few-shot):")
for m_name in LLMS:
    texts = S1[m_name, "few-shot"]
    own = [x.split("\nObservation:", 1)[1].split("\n")[0] for x in texts if "\nObservation:" in x]
    claims = [(c, a, m.group(1)) for (c, _), x in zip(ITEMS, texts) if "\nObservation:" in x
              for a, p in PAT.items() for m in [re.search(p, x.split("\nObservation:", 1)[1].split("\n")[0])] if m]
    matched = Counter(a for c, a, v in claims if norm(v) == norm(c[a]))
    print(f"   {m_name:<22} menulis 'Observation:' sendiri {len(own)}/{N_KB} | nilai fakta di dalamnya {len(claims)}, "
          f"yang cocok dengan ensiklopedia {sum(matched.values())} {dict(matched)}")
    if own:
        print(f"   {'':<22} contoh: {own[0][:120]!r}")

print("\n(b) langkah jawaban setelah aksi oracle dan Observation yang benar (48 token tanpa stop, lalu dipotong):")
S2 = {}
for m_name, make in LLMS.items():
    llm = make(48)
    S2[m_name] = [step_text(llm, "few-shot", QTXT[t].format(c["city"]), [(oracle_action(c), flat(c))]) for c, t in ITEMS]
    for lab, stops in [("stop buku", STOP_BOOK), ("stop buku + \\nQuestion:", STOP_PLUS)]:
        parsed = [parse(cut(x, stops)) for x in S2[m_name]]
        fin = [r for r in parsed if isinstance(r, AgentFinish)]
        clean = sum("\n" not in r.return_values["output"].strip() for r in fin)
        right = sum(isinstance(r, AgentFinish) and mentions(r.return_values["output"].split("\n")[0], c[t])
                    for r, (c, t) in zip(parsed, ITEMS))
        print(f"   {m_name:<22} {lab:<24} jawaban akhir terurai {len(fin):>2}/{N_KB} (satu baris {clean:>2}) | sesuai kunci {right:>2}/{N_KB}")
    print(f"   {'':<22} teks yang lanjut menulis '\\nQuestion:' {sum(chr(10) + 'Question:' in x for x in S2[m_name])}/{N_KB}")

(a) langkah aksi tanpa stop sequence (template few-shot):
   GPT-2                  menulis 'Observation:' sendiri 30/30 | nilai fakta di dalamnya 70, yang cocok dengan ensiklopedia 3 {'country': 1, 'river': 2}
                          contoh: ' Ostmund is a city in the city of Ostmund. Ostmund lies on the river Ostmund. The population of Ostmund is 5,857. The ma'
   Qwen2.5-1.5B-Instruct  menulis 'Observation:' sendiri 30/30 | nilai fakta di dalamnya 100, yang cocok dengan ensiklopedia 6 {'river': 3, 'country': 3}
                          contoh: ' Ostmund is a city in Dranvel. Ostmund was founded in 1153. Ostmund lies on the river Tornis. The population of Ostm'

(b) langkah jawaban setelah aksi oracle dan Observation yang benar (48 token tanpa stop, lalu dipotong):
   GPT-2                  stop buku                jawaban akhir terurai 10/30 (satu baris  0) | sesuai kunci  4/30
   GPT-2                  stop buku + \nQuestion:  jawaban akhir terurai 30/30 (satu baris 30) | sesuai

**(a) Dengan template few-shot dan tanpa stop, kedua model menulis hasil alatnya sendiri** — di 30 dari 30 soal, langsung setelah *Action Input*, meniru baris Observation di contoh. Observation buatan model itu meniru kalimat ensiklopedia dengan meyakinkan, tetapi isinya bukan fakta kota yang ditanya. Contoh Qwen untuk Ostmund — berdiri 1153 di Dranvel, di tepi sungai Tornis — ternyata fakta kota contoh pertama di prompt (Lumwick) dengan nama kotanya ditukar; fakta Ostmund sebenarnya 1886 di Ostravia. Dari 70 nilai fakta yang ditulis GPT-2 hanya 3 yang cocok dengan kota yang ditanya, dan dari 100 nilai Qwen hanya 6 — semuanya negara atau sungai, yang pilihannya sedikit (6 negara, 8 sungai) sehingga mudah cocok secara kebetulan. Karena itulah stop `\nObservation:` wajib. Tanpa itu, loop ReAct berubah menjadi model yang menjawab dirinya sendiri, dan model instruksi pun tidak kebal.

**(b) Di langkah jawaban, kebutuhan kedua model berbeda.** Qwen berhenti sendiri setelah *Final Answer* karena dilatih mengakhiri giliran chat-nya, jadi stop buku sudah cukup: 30 dari 30 jawaban satu baris dan sesuai kunci. GPT-2 tidak mengenal giliran; setelah *Final Answer* ia terus menulis soal baru seperti di contoh (30 dari 30 teks memuat `\nQuestion:`). Dengan stop buku saja, hanya 10 dari 30 teks terurai sebagai jawaban akhir dan tak satu pun satu baris — soal karangan berikutnya ikut masuk ke jawaban. Menambahkan `\nQuestion:` sebagai stop memberi 30 jawaban bersih, 12 di antaranya benar. Eksperimen selanjutnya memakai stop gabungan ini.

## Evaluasi ReAct

Buku mendaftar penerapan ReAct — tanya jawab dengan pengetahuan eksternal, navigasi web, kendali aplikasi lewat API, robotika, dan pemecahan masalah bertahap — lalu metrik evaluasinya: tingkat keberhasilan, efisiensi (jumlah langkah atau waktu), akurasi penalaran, relevansi aksi, **pemanfaatan pengamatan** (*observation utilization*), dan analisis galat, dengan teknik seperti metrik otomatis dan ablasi.

Buku tidak memberi angka. Sebagai gambaran dari makalah ReAct asli (PaLM-540B, dari literatur): di HotpotQA — tanya jawab yang butuh menggabungkan beberapa artikel Wikipedia — ReAct hanya sekitar 2 poin di atas versi tanpa *Thought* (*Act-only*), dan sedikit di bawah *chain-of-thought* yang tidak memakai alat. Keunggulannya baru terlihat ketika kegagalan dibedah: 56% kegagalan *chain-of-thought* berasal dari fakta karangan, sedangkan pada ReAct 0%.

UTH-1 sudah mengukur relevansi aksi. Metrik yang paling dekat dengan janji ReAct — jawaban yang bersandar pada pengamatan — kita ukur sekarang.

### UTH-3 — Apakah Observation benar-benar dibaca?

Di kota fiktif, jawaban benar hanya ada di Observation. Tetapi jawaban yang *cocok* dengan Observation belum tentu hasil membaca: model bisa saja menyalin nilai mana pun dari teks itu. Karena itu isi Observation dimanipulasi. Tiga kondisi pertama memakai langkah 1 oracle, template few-shot, 32 token, dan stop buku + `\nQuestion:`:

- **a. dokumen benar** — dokumen kota yang ditanya;
- **b. dokumen kota lain** — meniru pencarian yang salah ambil; model yang membaca Observation akan menjawab dengan nilai kota pengganti;
- **c. atribut dihapus** — dokumen yang benar tanpa kalimat yang memuat jawabannya; jawaban yang jujur adalah "tidak tahu";
- **d. tanpa alat** — prompt *closed-book* (tanpa alat dan tanpa Observation) dengan tiga contoh tanya-jawab yang sama, 24 token, berhenti di baris baru.

Baris pertama *Final Answer* digolongkan dengan urutan pemeriksaan berikut: kosong; menyebut **kunci**; **ikut Observation** (nilai kota pengganti); **atribut lain** dari dokumen; **nilai contoh** few-shot; **tidak tahu**; selebihnya **lain** (nilai karangan). Teks yang tak terurai dihitung **gagal urai**.

In [27]:
OTHER = [KB[(i + 1) % N_KB] for i in range(N_KB)]   # dokumen kota berikutnya: pencarian yang salah ambil
COND = {"a. dokumen benar": lambda i, c, t: (flat(c), c, ATTR),
        "b. dokumen kota lain": lambda i, c, t: (flat(OTHER[i]), OTHER[i], ATTR),
        "c. atribut dihapus": lambda i, c, t: (flat(c, drop=t), c, [x for x in ATTR if x != t])}
CLOSED = "Answer the following questions as best you can.\n\n" + "".join(
    f"Question: {QTXT[t].format(s['city'])}\nFinal Answer: {s[t]}\n\n" for s, t in zip(SHOTS, SHOT_TYPES))
ABSTAIN = re.compile(r"\b(unknown|no information|cannot|can't|don't know|not\b.{0,20}\b(mentioned|provided|available|found|known|given|specified|stated|listed))\b", re.I)


def category(ans, c, t, oc, shown):
    """Kategori saling lepas, diperiksa berurutan."""
    if not norm(ans).strip():
        return "kosong", None
    if mentions(ans, c[t]):
        return "kunci", t
    if oc is not c and mentions(ans, oc[t]):
        return "ikut Observation", t
    for x in shown:
        if x != t and mentions(ans, oc[x]):
            return "atribut lain", x
    if any(mentions(ans, s[x]) for s in SHOTS for x in ATTR):
        return "nilai contoh", None
    if ABSTAIN.search(ans):
        return "tidak tahu", None
    return "lain", None


t0 = time.perf_counter()
U3 = {}
for m_name, make in LLMS.items():
    llm = make(32)
    for cond, f in COND.items():
        rows = []
        for i, (c, t) in enumerate(ITEMS):
            obs, oc, shown = f(i, c, t)
            r = parse(step_text(llm, "few-shot", QTXT[t].format(c["city"]), [(oracle_action(c), obs)], stop=STOP_PLUS))
            ans = r.return_values["output"].split("\n")[0] if isinstance(r, AgentFinish) else None
            cat, pick = category(ans, c, t, oc, shown) if ans is not None else ("gagal urai", None)
            rows.append(dict(t=t, ans=ans, cat=cat, pick=pick))
        U3[m_name, cond] = rows
    closed_llm = make(24)
    rows = []
    for c, t in ITEMS:
        ans = closed_llm.invoke(CLOSED + f"Question: {QTXT[t].format(c['city'])}\nFinal Answer:", stop=["\n"]).content
        ans = ans.replace("Final Answer:", "").strip()
        cat, pick = category(ans, c, t, c, [])
        rows.append(dict(t=t, ans=ans, cat=cat, pick=pick))
    U3[m_name, "d. tanpa alat"] = rows
print(f"{len(U3)} kombinasi x {N_KB} soal, {time.perf_counter() - t0:.0f} s\n")
CATS = ["kunci", "ikut Observation", "atribut lain", "nilai contoh", "tidak tahu", "lain", "kosong", "gagal urai"]
print(f"{'model':<22} {'kondisi':<21}" + "".join(f"{k:>17}" for k in CATS))
for (m_name, cond), rows in U3.items():
    cnt = Counter(r["cat"] for r in rows)
    print(f"{m_name:<22} {cond:<21}" + "".join(f"{cnt[k]:>17}" for k in CATS))
print()
for m_name in LLMS:
    print(f"{m_name}, dokumen benar: atribut yang dipakai sebagai jawaban, per jenis pertanyaan")
    for t in TYPES:
        cnt = Counter(r["pick"] or r["cat"] for r in U3[m_name, "a. dokumen benar"] if r["t"] == t)
        print(f"   ditanya {t:<11} -> " + ", ".join(f"{k} {v}" for k, v in cnt.most_common()))
for m_name in LLMS:
    top = Counter((r["t"], r["ans"]) for r in U3[m_name, "c. atribut dihapus"]).most_common(4)
    print(f"{m_name}, atribut dihapus, jawaban tersering (jenis, jawaban): {top}")
for m_name in LLMS:
    print(f"{m_name}, contoh jawaban (dokumen benar | atribut dihapus | tanpa alat):",
          [(U3[m_name, 'a. dokumen benar'][k]['ans'], U3[m_name, 'c. atribut dihapus'][k]['ans'], U3[m_name, 'd. tanpa alat'][k]['ans'])
           for k in range(3)])

8 kombinasi x 30 soal, 317 s

model                  kondisi                          kunci ikut Observation     atribut lain     nilai contoh       tidak tahu             lain           kosong       gagal urai
GPT-2                  a. dokumen benar                    12                0               17                0                0                1                0                0
GPT-2                  b. dokumen kota lain                 0               11               17                0                0                2                0                0
GPT-2                  c. atribut dihapus                   0                0               28                0                0                2                0                0
GPT-2                  d. tanpa alat                        0                0                0               18                0               12                0                0
Qwen2.5-1.5B-Instruct  a. dokumen benar                    30    

**Qwen membaca Observation menurut pertanyaannya; GPT-2 menyalin dari Observation, tetapi tidak menurut pertanyaan.**

Dengan dokumen benar (a), Qwen menjawab 30 dari 30 dengan atribut yang ditanya — populasi untuk soal populasi, wali kota untuk soal wali kota. GPT-2 benar 12 dari 30, dan rinciannya menunjukkan sebabnya: 23 dari 30 jawabannya adalah tahun berdiri, apa pun pertanyaannya. Soal tahun benar semua (10 dari 10), soal populasi hanya 2 dari 10, dan soal wali kota 0 dari 10. Skor 12 itu hampir seluruhnya hasil kebiasaan menyalin tahun yang kebetulan tepat untuk sepertiga soal. Tanpa alat (d), keduanya 0 benar: GPT-2 menyalin nilai contoh (18 dari 30), sedangkan Qwen lebih sering mengarang (14) atau menyalin contoh (10) daripada mengaku tidak tahu (6).

Dua manipulasi berikutnya menguji asumsi bahwa Observation selalu benar dan lengkap:

- **Dokumen kota lain (b).** Qwen menjawab 30 dari 30 dengan nilai kota pengganti, padahal nama kota di dokumen berbeda dengan nama di soal. Kepatuhan pada Observation yang membuat kondisi (a) sempurna adalah juga yang membuat galat pencarian merambat utuh ke jawaban — *error propagation* dari daftar keterbatasan buku.
- **Atribut dihapus (c).** Jawaban yang jujur di sini adalah "tidak tahu", dan tidak satu model pun memberikannya. Qwen mengisi kekosongan dengan nilai dari contoh few-shot (16 soal; misalnya tahun 1459 dan wali kota "Tomas Mattrel", keduanya milik kota contoh Zunholt) atau dengan nilai karangan (12 soal; misalnya populasi "10,000", enam kali). GPT-2 menyalin atribut lain dari dokumen (28). Format *Final Answer* menuntut satu nilai, dan model memberinya.

Jadi "pemanfaatan pengamatan" bukan satu angka. Qwen lulus uji membaca dengan sempurna, tetapi skor itu tidak mengatakan apa pun tentang dokumen yang salah atau tidak lengkap — dua situasi yang justru sering terjadi pada alat sungguhan.

## Keamanan, kendali, dan keterbatasan

Buku mendaftar risiko agen ReAct — perilaku tak terduga, aksi yang berdampak nyata, bias, penyalahgunaan, dan akuntabilitas — serta mitigasinya: *sandboxing*, pengawasan manusia, aturan dan batasan, pemantauan dan audit, serta transparansi. Bagian ini di buku tidak berisi kode, tetapi notebook ini sudah memakai beberapa di antaranya:

- **Sandbox dan batasan.** Alat hanya membaca data lokal; kalkulator menolak apa pun selain angka dan operator, dan membatasi ukuran hasil. Di jalur utama, model tidak pernah mendapat akses ke shell, Python, atau jaringan; sel opsional Wikipedia adalah satu-satunya alat yang membaca dari internet.
- **Rem.** `max_iterations` dan penjaga konteks memastikan loop yang tersesat tetap berhenti.
- **Audit.** `CALLS` mencatat setiap panggilan model dan `TOOL_LOG` setiap eksekusi `Search`/`Calculator`; tabel biaya di bawah disusun dari log itu.

Keterbatasan yang didaftar buku — ketergantungan pada alat, galat yang merambat, batas token, biaya komputasi, dan sulitnya merancang prompt — sebagian sudah terlihat: ketergantungan pada alat di kondisi tanpa alat pada UTH-3 (0 benar), galat yang merambat di kondisi dokumen kota lain, dan sulitnya merancang prompt di UTH-1. Dua sisanya, biaya dan batas token, diukur dengan menjalankan loop penuh.

### UTH-4 — Loop penuh: keberhasilan, biaya, dan batas

Kali ini setiap model dijalankan di dalam `AgentExecutor` lengkap (`handle_parsing_errors=True`, `max_iterations` bawaan 15) dengan template few-shot, stop buku + `\nQuestion:`, dan 40 token per panggilan. Pembandingnya adalah **kebijakan oracle**: program deterministik — bukan model bahasa — yang membaca soal dan Observation dari prompt lalu menulis langkah yang benar, dan dijalankan lewat executor, prompt, parser, serta alat yang sama. Keberhasilannya memeriksa mesin agen; jumlah panggilan dan token-nya adalah biaya minimum loop ini.

In [28]:
ORACLE = []


class OraclePolicy(BaseChatModel):
    """Kebijakan sempurna deterministik, BUKAN model bahasa: membaca soal dan Observation dari prompt lalu menulis langkah yang benar."""

    plan: Any = None

    @property
    def _llm_type(self) -> str:
        return "oracle-policy"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        p = "\n".join(str(m.content) for m in messages)
        tail = p.split("Begin!\n", 1)[1]   # soal uji + scratchpad (contoh few-shot ada sebelum 'Begin!')
        question = tail.split("\n", 1)[0].removeprefix("Question: ")
        text = self.plan(question, re.findall(r"\nObservation: (.*)", tail))
        n_prompt = len(agent_tok(agent_llm().render(messages), add_special_tokens=False)["input_ids"])   # dihitung dengan tokenizer model agen
        ORACLE.append(dict(prompt_tokens=n_prompt, new_tokens=len(agent_tok(text)["input_ids"]), text=text, s=0.0, overflow=False))
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])


def kb_plan(question, observations):
    c = next(BYNAME[w] for w in re.findall(r"[A-Z]\w+", question) if w in BYNAME)
    t = next(t for t in TYPES if QTXT[t].format(c["city"]) == question)
    if not observations:
        return f"Thought: I need to look up {c['city']}.\nAction: Search\nAction Input: {c['city']}"
    m = re.search(PAT[t], observations[-1])
    return f"Thought: I now know the final answer\nFinal Answer: {m.group(1) if m else 'unknown'}"


def run_loop(model, prompt_tpl, tools_, question):
    ag = ({"input": lambda x: x["input"], "agent_scratchpad": lambda x: format_log_to_str(x["intermediate_steps"])}
          | prompt_tpl | model.bind(stop=STOP_PLUS) | ReActSingleInputOutputParser())
    ex = AgentExecutor(agent=ag, tools=tools_, handle_parsing_errors=True, return_intermediate_steps=True)   # max_iterations bawaan 15
    log = ORACLE if isinstance(model, OraclePolicy) else CALLS
    n0, k0, t0 = len(log), len(TOOL_LOG), time.perf_counter()
    try:
        out = ex.invoke({"input": question})["output"]
        why = "batas iterasi" if out.startswith("Agent stopped") else "selesai"
    except ContextOverflow:
        out, why = None, "konteks penuh"
    return dict(out=out, why=why, calls=[c for c in log[n0:] if not c["overflow"]], tools=TOOL_LOG[k0:], s=time.perf_counter() - t0)


def summarize(name, runs, gold):
    ok = [r["out"] is not None and r["why"] == "selesai" and mentions(r["out"].split("\n")[0], g) for r, g in zip(runs, gold)]
    why = Counter(r["why"] for r in runs)
    n_calls = [len(r["calls"]) for r in runs]
    tokens = [sum(c["prompt_tokens"] + c["new_tokens"] for c in r["calls"]) for r in runs]
    print(f"{name:<22} benar {sum(ok):>2}/{len(ok)} {ci(ok)} | selesai/batas iterasi/konteks penuh "
          f"{why['selesai']}/{why['batas iterasi']}/{why['konteks penuh']} | panggilan {np.mean(n_calls):.1f}/soal (maks {max(n_calls)}) | "
          f"token {np.mean(tokens):,.0f}/soal | {np.mean([r['s'] for r in runs]):.1f} s/soal")
    return ok


P_KB = TPL["few-shot"]
gold_kb = [c[t] for c, t in ITEMS]
t0 = time.perf_counter()
LOOP_KB = {name: [run_loop(make(40), P_KB, [search_tool], QTXT[t].format(c["city"])) for c, t in ITEMS] for name, make in LLMS.items()}
LOOP_KB["kebijakan oracle"] = [run_loop(OraclePolicy(plan=kb_plan), P_KB, [search_tool], QTXT[t].format(c["city"])) for c, t in ITEMS]
print(f"{N_KB} soal x {len(LOOP_KB)} kebijakan, {time.perf_counter() - t0:.0f} s (token GPT-2 dihitung dengan tokenizer GPT-2; lainnya dengan tokenizer {AGENT_LABEL})")
OK_KB = {name: summarize(name, runs, gold_kb) for name, runs in LOOP_KB.items()}
for t in TYPES:
    picks = Counter(category(r["out"].split("\n")[0], c, tt, c, ATTR)[1] or "lain"
                    for r, (c, tt) in zip(LOOP_KB["GPT-2"], ITEMS) if tt == t and r["out"] is not None)
    print(f"   loop GPT-2, ditanya {t:<11} -> atribut di jawaban: {dict(picks.most_common())}")

30 soal x 3 kebijakan, 213 s (token GPT-2 dihitung dengan tokenizer GPT-2; lainnya dengan tokenizer Qwen2.5-1.5B-Instruct)
GPT-2                  benar 12/30 [0.25; 0.58] | selesai/batas iterasi/konteks penuh 30/0/0 | panggilan 2.0/soal (maks 2) | token 1,104/soal | 1.4 s/soal
Qwen2.5-1.5B-Instruct  benar 30/30 [0.89; 1.00] | selesai/batas iterasi/konteks penuh 30/0/0 | panggilan 2.0/soal (maks 2) | token 1,162/soal | 5.7 s/soal
kebijakan oracle       benar 30/30 [0.89; 1.00] | selesai/batas iterasi/konteks penuh 30/0/0 | panggilan 2.0/soal (maks 2) | token 1,159/soal | 0.0 s/soal
   loop GPT-2, ditanya population  -> atribut di jawaban: {'founded': 6, 'river': 2, 'population': 2}
   loop GPT-2, ditanya founded     -> atribut di jawaban: {'founded': 10}
   loop GPT-2, ditanya mayor       -> atribut di jawaban: {'founded': 7, 'lain': 1, 'river': 1, 'population': 1}


Di ensiklopedia, loop penuh berjalan rapi untuk ketiganya: setiap soal selesai dalam tepat 2 panggilan (satu `Search`, satu *Final Answer*), tanpa menyentuh batas iterasi atau konteks. Kebijakan oracle benar 30 dari 30, jadi mesin agennya — executor, prompt, parser, alat — bekerja. Qwen juga 30 dari 30. GPT-2 benar 12 dari 30 (CI95 [0,25; 0,58]), jumlah yang sama dengan UTH-3 (a), dan rinciannya pun sama: 23 jawaban berisi tahun berdiri, apa pun pertanyaannya. Kebiasaan menyalin tahun terbawa utuh ke loop penuh.

Biayanya hampir sama untuk ketiganya, sekitar 1.100–1.160 token per soal, dan hampir semuanya prompt: instruksi dan tiga contoh dibaca di langkah 1, lalu dibaca ulang di langkah 2 bersama aksi dan Observation. Di tugas satu pencarian, jawaban yang benar dan yang salah sama mahalnya. Perbedaan biaya baru muncul ketika loop tidak tahu kapan berhenti, seperti di soal aritmetika berikut. (Token GPT-2 dihitung dengan tokenizer GPT-2, lainnya dengan tokenizer Qwen; bandingkan jumlah panggilannya lebih dulu.)

Di ensiklopedia, satu pencarian sudah cukup. Soal aritmetika memisahkan dua kemampuan yang di sana bertumpuk: **memilih aksi** dan **memakai hasilnya**. Kita ambil 20 soal MultiArith (Roy & Roth, 2015) yang masing-masing butuh tepat dua operasi, dengan alat `Calculator` dan tiga contoh few-shot dua langkah. Berkasnya diunduh dari repositori publik dan diverifikasi dengan SHA-256.

In [29]:
URL = "https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/main/dataset/MultiArith/MultiArith.json"
SHA256 = "dc9db1b6cc68b1ecc2dec830d15aaf8c9a0932aa31d5ffce88ef3628179d36f4"
path = WORK / "MultiArith.json"
if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != SHA256:
    tmp = path.with_suffix(".tmp")
    urllib.request.urlretrieve(URL, tmp)
    tmp.replace(path)   # berkas sebagian dari unduhan yang terputus tidak pernah menempati nama akhir
raw = path.read_bytes()
assert hashlib.sha256(raw).hexdigest() == SHA256, "isi MultiArith.json berbeda dari versi yang diuji (sumber di GitHub berubah)"
multiarith = json.loads(raw)
OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "*": lambda a, b: a * b, "/": lambda a, b: a / b if b else float("nan")}


def two_steps(equation):
    """'X=((a op b) op c)' atau 'X=(a op (b op c))' -> dua langkah (a, op, b, hasil)."""
    e = equation.replace("X=", "").replace(" ", "")
    num = r"([0-9]+\.?[0-9]*)"
    m1 = re.fullmatch(rf"\(\({num}([-+*/]){num}\)([-+*/]){num}\)", e)
    m2 = re.fullmatch(rf"\({num}([-+*/])\({num}([-+*/]){num}\)\)", e)
    if m1:
        a, o1, b, o2, c = m1.groups()
        a, b, c = float(a), float(b), float(c)
        r1 = OPS[o1](a, b)
        return [(a, o1, b, r1), (r1, o2, c, OPS[o2](r1, c))]
    if m2:
        a, o2, b, o1, c = m2.groups()
        a, b, c = float(a), float(b), float(c)
        r1 = OPS[o1](b, c)
        return [(b, o1, c, r1), (a, o2, r1, OPS[o2](a, r1))]
    return None


MA = {}
for d in multiarith:
    steps = two_steps(d["lEquations"][0])
    q = re.sub(r"\s+", " ", d["sQuestion"]).strip()
    if steps and abs(steps[-1][3] - float(d["lSolutions"][0])) < 1e-9:
        MA.setdefault(q, steps)   # teks soal ganda disimpan sekali
MA = list(MA.items())
order = random.Random(0).sample(range(len(MA)), len(MA))
MA_SHOTS = [MA[i] for i in order[:3]]
MA_TEST = [MA[i] for i in order[3:] if MA[i][1] not in [st for _, st in MA_SHOTS]][:N_MA]
print(f"MultiArith: {len(multiarith)} soal, {len(MA)} teks unik dengan dua langkah | 3 contoh few-shot, {len(MA_TEST)} soal uji")
print("soal uji 1:", MA_TEST[0][0])

MultiArith: 600 soal, 580 teks unik dengan dua langkah | 3 contoh few-shot, 20 soal uji
soal uji 1: Kaleb was collecting cans for recycling. On Saturday he filled 5 bags up and on Sunday he filled 5 more bags. If each bag had 4 cans in it, how many cans did he pick up total?


Selain loop penuh, ada tiga pembanding satu panggilan untuk setiap model:

- **aksi oracle + Observation asli** — kedua aksi kalkulator diberikan, model hanya menulis langkah terakhir;
- **aksi oracle + Observation +10** — sama, tetapi hasil kalkulator terakhir digeser +10, seolah alatnya keliru;
- **tanpa alat** — contoh yang sama sebagai pasangan soal-jawaban.

In [30]:
def step_s(s):
    return f"{fmt(s[0])} {s[1]} {fmt(s[2])}"


def ma_trace(q, st):
    text = f"Question: {q}\n"
    for k, s in enumerate(st):
        text += f"Thought: {'First' if k == 0 else 'Next'} I compute {step_s(s)}.\nAction: Calculator\nAction Input: {step_s(s)}\nObservation: {fmt(s[3])}\n"
    return text + f"Thought: I now know the final answer\nFinal Answer: {fmt(st[-1][3])}\n\n"


P_MA = make_prompt(HEAD + esc("".join(ma_trace(q, st) for q, st in MA_SHOTS)) + "Begin!\nQuestion: {input}\n{agent_scratchpad}", calc_tool)
MA_GOLD = dict(MA_TEST)


def ma_plan(q, obs):
    (a, o1, b, r1), (x, o2, y, r2) = MA_GOLD[q]
    if not obs:
        return f"Thought: First I compute {step_s(MA_GOLD[q][0])}.\nAction: Calculator\nAction Input: {step_s(MA_GOLD[q][0])}"
    if len(obs) == 1:
        expr = f"{obs[0]} {o2} {fmt(y)}" if x == r1 else f"{fmt(x)} {o2} {obs[0]}"   # langkah 2 memakai hasil Observation
        return f"Thought: Next I compute {expr}.\nAction: Calculator\nAction Input: {expr}"
    return f"Thought: I now know the final answer\nFinal Answer: {obs[-1]}"


gold_ma = [fmt(st[-1][3]) for _, st in MA_TEST]
t0 = time.perf_counter()
LOOP_MA = {name: [run_loop(make(40), P_MA, [calc_tool], q) for q, _ in MA_TEST] for name, make in LLMS.items()}
LOOP_MA["kebijakan oracle"] = [run_loop(OraclePolicy(plan=ma_plan), P_MA, [calc_tool], q) for q, _ in MA_TEST]
print(f"loop penuh: {N_MA} soal x {len(LOOP_MA)} kebijakan, {time.perf_counter() - t0:.0f} s")
OK_MA = {name: summarize(name, runs, gold_ma) for name, runs in LOOP_MA.items()}

# Pembanding satu panggilan: aksi oracle + Observation asli, aksi oracle + Observation terakhir digeser +10, dan tanpa alat
CLOSED_MA = "Answer the following questions as best you can.\n\n" + "".join(
    f"Question: {q}\nFinal Answer: {fmt(st[-1][3])}\n\n" for q, st in MA_SHOTS)
print()
for m_name, make in LLMS.items():
    llm = make(32)
    real, shifted, closed = [], [], []
    for q, st in MA_TEST:
        obs, steps = [], []
        for k in range(2):
            e = ma_plan(q, obs).split("Action Input: ")[1]
            o = calculator(e)
            obs.append(o)
            steps.append((AgentAction("Calculator", e, f"Thought: {'First' if k == 0 else 'Next'} I compute {e}.\nAction: Calculator\nAction Input: {e}"), o))
        fake = fmt(float(obs[-1]) + 10)
        for store, last in [(real, obs[-1]), (shifted, fake)]:
            msgs = P_MA.invoke({"input": q, "agent_scratchpad": format_log_to_str(steps[:1] + [(steps[1][0], last)])}).to_messages()
            r = parse(llm.invoke(msgs, stop=STOP_PLUS).content)
            store.append((r.return_values["output"].split("\n")[0] if isinstance(r, AgentFinish) else None, last))
        closed.append(make(12).invoke(CLOSED_MA + f"Question: {q}\nFinal Answer:", stop=["\n"]).content.replace("Final Answer:", "").strip())
    print(f"{m_name:<22} aksi oracle + Observation asli : jawaban = Observation terakhir {sum(a is not None and mentions(a, o) for a, o in real):>2}/{N_MA} "
          f"| = kunci {sum(a is not None and mentions(a, g) for (a, _), g in zip(real, gold_ma)):>2}/{N_MA}")
    print(f"{'':<22} aksi oracle + Observation +10 : jawaban = angka yang digeser {sum(a is not None and mentions(a, o) for a, o in shifted):>2}/{N_MA} "
          f"| = kunci {sum(a is not None and mentions(a, g) for (a, _), g in zip(shifted, gold_ma)):>2}/{N_MA}")
    print(f"{'':<22} tanpa alat (closed-book)      : = kunci {sum(mentions(a, g) for a, g in zip(closed, gold_ma)):>2}/{N_MA}")
print()
for name in LLMS:
    first = [r["tools"][0][1].replace(" ", "") if r["tools"] else None for r in LOOP_MA[name]]
    right1 = sum(f == step_s(st[0]).replace(" ", "") for f, (_, st) in zip(first, MA_TEST))
    repeat = sum(len(r["tools"]) > len({x for _, x in r["tools"]}) for r in LOOP_MA[name])
    used = Counter(len(r["tools"]) for r in LOOP_MA[name])
    print(f"{name:<22} aksi pertama = langkah 1 kunci {right1:>2}/{N_MA} | run dengan input alat berulang {repeat:>2}/{N_MA} | "
          f"jumlah eksekusi alat per soal {dict(sorted(used.items()))}")
print("\ncontoh input Calculator, soal 1-2:")
for name in LLMS:
    print(f"   {name:<22}", [[x.strip() for _, x in r["tools"][:5]] for r in LOOP_MA[name][:2]])
print(f"   {'kunci':<22}", [[step_s(s) for s in st] for _, st in MA_TEST[:2]])

loop penuh: 20 soal x 3 kebijakan, 333 s
GPT-2                  benar  0/20 [0.00; 0.16] | selesai/batas iterasi/konteks penuh 8/9/3 | panggilan 9.4/soal (maks 15) | token 6,729/soal | 7.2 s/soal
Qwen2.5-1.5B-Instruct  benar 16/20 [0.58; 0.92] | selesai/batas iterasi/konteks penuh 19/1/0 | panggilan 3.5/soal (maks 15) | token 2,476/soal | 9.5 s/soal
kebijakan oracle       benar 20/20 [0.84; 1.00] | selesai/batas iterasi/konteks penuh 20/0/0 | panggilan 3.0/soal (maks 3) | token 1,870/soal | 0.0 s/soal

GPT-2                  aksi oracle + Observation asli : jawaban = Observation terakhir 19/20 | = kunci 19/20
                       aksi oracle + Observation +10 : jawaban = angka yang digeser 20/20 | = kunci  0/20
                       tanpa alat (closed-book)      : = kunci  0/20
Qwen2.5-1.5B-Instruct  aksi oracle + Observation asli : jawaban = Observation terakhir 20/20 | = kunci 20/20
                       aksi oracle + Observation +10 : jawaban = angka yang digeser 20/20 | = kunci

**Memakai hasil alat: ya, termasuk ketika hasilnya keliru.** Bila kedua aksi kalkulator diberikan oracle, Qwen menulis jawaban yang sama dengan Observation terakhir di 20 dari 20 soal dan GPT-2 di 19 dari 20. Ketika Observation terakhir digeser +10, seolah kalkulatornya keliru, *keduanya* menyalin angka yang salah itu 20 dari 20 kali, dan tak satu pun benar. Yang terukur adalah menyalin angka terakhir di *scratchpad* ke *Final Answer*, bukan memeriksa apakah hasil alat masuk akal. Tanpa alat, keduanya hampir tak pernah benar (GPT-2 0, Qwen 2 dari 20).

**Memilih aksi: di sinilah kedua model berpisah.** Loop penuh Qwen benar 16 dari 20 (CI95 [0,58; 0,92]); di 15 soal ia memanggil kalkulator tepat dua kali, seperti di contoh. GPT-2 benar 0 dari 20: separuh run-nya mengulang input alat yang persis sama, dan 12 run tidak berhenti sendiri — 9 dihentikan `max_iterations=15`, 3 oleh konteks penuh. Di soal 2, GPT-2 menghitung `25 + 15`, `40 - 48`, `8 * 8`, … tanpa arah, sedangkan Qwen menulis `25 - 15` lalu `10 / 2`, persis kunci.

**Biayanya mengikuti kegagalan.** Kebijakan oracle butuh 3 panggilan dan sekitar 1.870 token per soal. Qwen butuh 3,5 panggilan dan 2.476 token, sekitar 1,3 kali oracle. GPT-2 butuh 9,4 panggilan dan 6.729 token, sekitar 3,6 kali oracle (tokenizer-nya berbeda, jadi rasio ini perkiraan). ReAct memindahkan hitungan dari model ke alat, tetapi memilih aksi tetap pekerjaan model; ketika itu gagal, alat yang benar tidak menolong, dan loop yang tidak tahu kapan berhenti membayar paling mahal.

Model tidak dibutuhkan lagi. Selain `base_model` dan `agent_model`, setiap `LocalChat` dan setiap agen yang membungkusnya masih memegang referensi ke model, jadi semuanya dilepas bersama sebelum memori GPU dikosongkan.

In [31]:
# setiap LocalChat, setiap agen yang membungkusnya, dan variabel loop `model` dari sel pemuatan ikut memegang referensi ke model
for name in ["model", "llm", "llm_with_stop", "closed_llm", "agent", "rebuilt", "gpt2_agent", "react", "react_wiki",
             "lcel_agent", "agent_executor", "base_model", "agent_model"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"memori GPU setelah model dilepas: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
print(f"total panggilan model: {len(CALLS)} | total eksekusi alat: {len(TOOL_LOG)}")

memori GPU setelah model dilepas: 0.01 GiB
total panggilan model: 948 | total eksekusi alat: 426


## Ringkasan

ReAct membuat model bergantian menulis *Thought* dan *Action*, sementara program menjalankan alat dan mengembalikan *Observation*, sampai muncul *Final Answer*. Yang kita lihat ketika pola itu dijalankan dengan dua model kecil:

- **Mesinnya sederhana; API LangChain-nya sudah berpindah.** Loop ReAct bisa ditulis dalam 20-an baris Python. Semua API agen yang dipakai buku kini ada di `langchain-classic` dan `langchain-community`, dan `initialize_agent` ditandai usang. Beberapa listing juga punya masalah sendiri: nama enum yang terpotong spasi, prompt kustom yang tak pernah dipasang dan memakai nama alat yang salah, deskripsi alat yang diubah setelah agen dibuat, dan `OpenAI` yang tak pernah diimpor.
- **Qwen cukup diberi instruksi; GPT-2 butuh contoh.** Dengan template buku, Qwen menulis aksi yang sah 30 dari 30, GPT-2 0 dari 30. Dengan tiga contoh, GPT-2 juga 30 dari 30.
- **Stop sequence wajib.** Tanpa `\nObservation:` (template few-shot), kedua model menulis Observation sendiri di 30 dari 30 soal, dan isinya hampir tak pernah fakta kota yang ditanya — sebagian malah disalin dari kota contoh di prompt.
- **Membaca Observation tidak sama dengan memeriksanya.** Dengan dokumen benar, Qwen benar 30 dari 30 dan GPT-2 12 dari 30 (kebanyakan karena menyalin tahun). Tetapi Qwen mengikuti dokumen kota lain 30 dari 30 kali, mengisi jawaban yang dihapus dengan nilai contoh atau karangan alih-alih mengaku tidak tahu, dan kedua model menyalin hasil kalkulator yang digeser +10 di 20 dari 20 soal.
- **Menyediakan alat tidak sama dengan memakainya.** Di dua demo pada run validasi, Qwen melewatkan `Calculator` dan salah berhitung di kepala. Di MultiArith, loop Qwen benar 16 dari 20, GPT-2 0 dari 20.
- **Biaya ditentukan oleh kapan loop berhenti.** Di soal satu pencarian semua kebijakan butuh sekitar 1.100 token per soal; di aritmetika, loop GPT-2 menghabiskan rata-rata sekitar 3,6 kali biaya oracle, dan 12 dari 20 run-nya baru berhenti karena rem iterasi atau konteks.

Buku menutup bab dengan ReAct sebagai pendahulu kerangka yang lebih maju seperti ReWOO (*Reasoning WithOut Observation*; Xu dkk., 2023), topik bab berikutnya. ReWOO menulis seluruh rencana pemanggilan alat sebelum alat dijalankan, dan makalahnya melaporkan efisiensi token sekitar 5 kali lipat di HotpotQA (dari literatur). UTH-4 memperlihatkan dari mana biaya ReAct berasal — setiap iterasi membaca ulang seluruh prompt — sehingga pertanyaan untuk bab berikutnya jelas: berapa yang bisa dihemat bila rencana ditulis sekali jalan?